# VP-RMT-SVD — notebook lengkap, folder sudah diperbaiki

Folder dataset sudah diatur ke **`/mnt/g/SVD/dataset`**, sesuai lokasi yang kamu
berikan. Nama file yang dipakai: **`health.csv`**, **`toys.csv`**, **`VideoGames.csv`**.

**Cara menjalankan:** buka notebook ini di kernel Python/Linux yang sama dengan
notebook sebelumnya, pilih **Restart Kernel**, lalu **Run All / Jalankan Semua**.
Semua kode tersedia dalam notebook ini; tidak perlu menyalin sel tambalan atau
mengimpor modul dari ZIP. Pengaturan folder hanya ada pada sel pertama.

Mode utama adalah **paper**: 3 dataset, 5 seed, 12 trial per komponen tunable,
200 permutations utama, full-item evaluation, baseline modern, matched controls,
sensitivitas dan statistik. Proses ini mencakup seluruh rancangan eksperimen.
Quick check awal hanya memverifikasi evaluator/solver; hasil paper dihitung dari
CSV asli ketika kamu menjalankan sel eksperimen.


In [1]:
from __future__ import annotations
from pathlib import Path

# Satu-satunya pengaturan folder data; sudah sesuai lokasi dataset kamu.
DATA_DIR = "/mnt/g/SVD/dataset"

# Hasil berada di /mnt/g/SVD/vp_rmt_reviewer_revision_runs.
OUTPUT_ROOT = str(Path(DATA_DIR).parent / "vp_rmt_reviewer_revision_runs")

print("Folder dataset:", DATA_DIR)
print("Folder hasil  :", OUTPUT_ROOT)


Folder dataset: /mnt/g/SVD/dataset
Folder hasil  : /mnt/g/SVD/vp_rmt_reviewer_revision_runs


## Seluruh implementasi

Jalankan sel berikut tanpa menambah tambalan. Sel ini memuat preprocessing,
candidate banks, evaluator, weighting, null spectrum, shrinkage, BPR/LightGCN/
LightGCL/iALS/EASE/PureSVD, tuning, fusion, NoGate, sensitivitas, synthetic recovery,
paired statistics, reviewer matrix, integrity checks dan pembuatan laporan.
Kode dapat dibuka dari sel yang dilipat jika perlu diperiksa.


In [2]:
"""VP-RMT reviewer revision. All model selection uses validation, never test.

This is an integration/adaptation study, not a new RMT theorem. The conditional
fixed-mask residual null assumes approximately exchangeable residual values;
the implicit null is a degree-stub multigraph null and is a separate experiment.
"""

import copy
import hashlib
import itertools
import json
import math
import os
import platform
import random
import time
import uuid
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path, PureWindowsPath

import numpy as np
import pandas as pd
import scipy
import scipy.linalg
import scipy.sparse as sp
from scipy.sparse.linalg import svds
from scipy.stats import rankdata, t as student_t
from sklearn.utils.extmath import randomized_svd
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
except ImportError:
    torch = None
    nn = None


DATASET_SPECS = {
    'health': dict(filename='health.csv', order='user_item', category='Health and Personal Care',
                   version_hint='Amazon 2014', expected_raw_rows=2982326,
                   source_url='https://cseweb.ucsd.edu/~jmcauley/datasets/amazon/links.html',
                   download_url='https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/ratings_Health_and_Personal_Care.csv'),
    'toys': dict(filename='toys.csv', order='user_item', category='Toys and Games',
                 version_hint='Amazon 2014', expected_raw_rows=2252771,
                 source_url='https://cseweb.ucsd.edu/~jmcauley/datasets/amazon/links.html',
                 download_url='https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/ratings_Toys_and_Games.csv'),
    'videogames': dict(filename='VideoGames.csv', order='user_item', category='Video Games',
                       version_hint='Amazon 2014', expected_raw_rows=1324753,
                       source_url='https://cseweb.ucsd.edu/~jmcauley/datasets/amazon/links.html',
                       download_url='https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/ratings_Video_Games.csv'),
}


@dataclass
class Config:
    mode: str = 'paper'
    data_dir: str = DATA_DIR
    # Optional exact paths, useful when category files have different names.
    # Relative paths resolve under data_dir; absolute paths override that folder.
    data_files: dict = field(default_factory=dict)
    output_root: str = OUTPUT_ROOT
    datasets: tuple = ('health', 'toys', 'videogames')
    seeds: tuple = (2021, 2022, 2023, 2024, 2025)
    tuning_seed: int = 7777
    split_seed: int = 77123
    candidate_seed: int = 88421
    min_user: int = 11
    min_item: int = 11
    valid_model_fraction: float = 0.60
    ks: tuple = (5, 10, 20)
    primary_k: int = 10
    n_trials: int = 12
    dim: int = 64
    max_epochs: int = 200
    eval_every: int = 5
    patience: int = 6
    batch: int = 2048
    device: str = 'auto'
    max_rank: int = 96
    bias_reg: float = 10.0
    vp_power: float = 0.5
    vp_smooth: float = 1.0
    factor_clip: tuple = (0.01, 0.99)
    edge_quantile: float = 0.95
    permutations: int = 200
    permutation_grid: tuple = (20, 50, 100, 200, 500)
    permutation_repeats: int = 5
    svd_tol: float = 1e-7
    svd_verify_tol: float = 1e-4
    fusion_grid: tuple = (-1., -.5, -.25, -.1, 0., .1, .25, .5, 1.)
    gate_delta: float = 0.0005
    gate_grid: tuple = (0., 0.0001, 0.0005, 0.001, 0.002)
    negative_grid: tuple = (100, 200, 500, 1000)
    candidate_repeats: int = 3
    bootstrap_reps: int = 2000
    randomization_reps: int = 5000
    synthetic_repeats: int = 10
    synthetic_shape: tuple = (120, 160)
    weighting_grid: tuple = (0., .25, .5, .75, 1.)
    run_sensitivities: bool = True
    run_synthetic: bool = True
    # Fill these from the actual download record; hints/counts are not proof.
    source_confirmations: dict = field(default_factory=dict)
    public_repository_url: str = ''
    jcta_full_name: str = ''

    def resolved(self):
        c = copy.deepcopy(self)
        if c.mode == 'smoke':
            c.seeds = (2024, 2025)
            c.min_user = c.min_item = 3
            c.n_trials = 2
            c.dim = 8
            c.max_epochs = 3
            c.eval_every = 1
            c.patience = 2
            c.batch = 64
            c.max_rank = 8
            c.permutations = 4
            c.permutation_grid = (2, 4, 8)
            c.permutation_repeats = 2
            c.fusion_grid = (-.5, 0., .5)
            c.gate_grid = (0., .0005, .002)
            c.negative_grid = (5, 15, 1000)
            c.candidate_repeats = 2
            c.bootstrap_reps = c.randomization_reps = 100
            c.synthetic_repeats = 2
            c.synthetic_shape = (24, 32)
            c.weighting_grid = (0., .5, 1.)
        if c.mode not in ('paper', 'smoke'):
            raise ValueError('mode must be paper or smoke')
        if not 0 < c.valid_model_fraction < 1:
            raise ValueError('validation partition fraction must be in (0,1)')
        if not 0 < c.edge_quantile < 1 or c.permutations < 2:
            raise ValueError('invalid permutation/quantile settings')
        if c.primary_k not in c.ks:
            raise ValueError('primary_k must be listed in ks')
        if not isinstance(c.data_files,dict):
            raise TypeError('data_files must be a dictionary: dataset key -> CSV path')
        if not c.datasets or len(set(c.datasets))!=len(c.datasets) or set(c.datasets)-set(DATASET_SPECS):
            raise ValueError('datasets must contain unique known keys: health, toys, videogames')
        if set(c.data_files)-set(DATASET_SPECS):
            raise ValueError('Unknown data_files key; use health, toys, videogames')
        return c


def jsonable(v):
    if isinstance(v, (np.integer, np.floating)):
        return v.item()
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, Path):
        return str(v)
    raise TypeError(type(v).__name__)


def sha_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def array_hash(*arrays):
    h = hashlib.sha256()
    for a in arrays:
        a = np.ascontiguousarray(a)
        h.update(str((a.shape, a.dtype)).encode())
        h.update(a.tobytes())
    return h.hexdigest()


class Recorder:
    def __init__(self, cfg):
        self.cfg = cfg
        run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
        self.root = Path(cfg.output_root) / run_id
        self.root.mkdir(parents=True, exist_ok=False)
        self.rows = defaultdict(list)
        self.dump_json('resolved_config.json', asdict(cfg))
        import importlib.metadata
        versions = {}
        for p in ('numpy', 'pandas', 'scipy', 'scikit-learn', 'torch', 'matplotlib'):
            try:
                versions[p] = importlib.metadata.version(p)
            except importlib.metadata.PackageNotFoundError:
                versions[p] = 'not_installed'
        self.dump_json('environment.json', dict(python=platform.python_version(),
            platform=platform.platform(), cpu=platform.processor(), versions=versions,
            cuda_available=bool(torch is not None and torch.cuda.is_available()),
            cuda_device_name=torch.cuda.get_device_name(0) if torch is not None and torch.cuda.is_available() else None,
            torch_threads=torch.get_num_threads() if torch is not None else None))
        (self.root / 'requirements-executed.txt').write_text('\n'.join(
            f'{k}=={v}' for k, v in versions.items() if v != 'not_installed') + '\n')

    def add(self, name, **row):
        row.setdefault('evidence_mode', self.cfg.mode)
        self.rows[name].append(row)

    def dump_json(self, name, obj):
        (self.root / name).write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')

    def save(self):
        for name, rows in self.rows.items():
            if rows:
                pd.DataFrame(rows).to_csv(self.root / f'{name}.csv', index=False)

    def table(self, name):
        return pd.DataFrame(self.rows[name])


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        # Sparse CUDA kernels may not be bitwise deterministic; log platform.
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def kcore(df, ku, ki):
    stages = []
    while True:
        n = len(df)
        u = df.user_raw.value_counts()
        df = df[df.user_raw.isin(u[u >= ku].index)]
        i = df.item_raw.value_counts()
        df = df[df.item_raw.isin(i[i >= ki].index)]
        stages.append(dict(rows=len(df), users=df.user_raw.nunique(), items=df.item_raw.nunique()))
        if len(df) == n or len(df) == 0:
            break
    if len(df) == 0:
        raise ValueError('Dataset collapsed under declared k-core; no silent filter changes.')
    return df.copy(), stages


def prepare_frame(raw, name, cfg, rec=None, provenance=None):
    raw = raw.copy()
    raw['_row'] = np.arange(len(raw))
    original_n = len(raw)
    for col in ('user_raw', 'item_raw'):
        raw[col] = raw[col].astype('string').str.strip()
    raw['rating'] = pd.to_numeric(raw.rating, errors='coerce')
    raw['timestamp'] = pd.to_numeric(raw.timestamp, errors='coerce')
    valid = raw[['user_raw', 'item_raw', 'rating', 'timestamp']].notna().all(axis=1)
    valid &= raw.user_raw.ne('') & raw.item_raw.ne('') & raw.rating.between(1, 5)
    raw = raw.loc[valid].copy()
    raw['timestamp'] = raw.timestamp.astype('int64')
    # Unique user-item edges, latest event retained. This changes old outputs:
    # the paper must report the new preprocessing, not reuse old numeric tables.
    raw = raw.sort_values(['timestamp', '_row'], kind='stable')
    cleaned_n = len(raw)
    raw = raw.drop_duplicates(['user_raw', 'item_raw'], keep='last')
    unique_n = len(raw)
    raw, stages = kcore(raw, cfg.min_user, cfg.min_item)
    users, items = sorted(raw.user_raw.unique()), sorted(raw.item_raw.unique())
    umap, imap = {v:k for k,v in enumerate(users)}, {v:k for k,v in enumerate(items)}
    raw['user'] = raw.user_raw.map(umap).astype('int64')
    raw['item'] = raw.item_raw.map(imap).astype('int64')
    raw = raw.sort_values(['user', 'timestamp', 'item', '_row'], kind='stable').reset_index(drop=True)
    g = raw.groupby('user', sort=False)
    pos, size = g.cumcount(), g.item.transform('size')
    if size.min() < 3:
        raise ValueError('LOO requires at least three unique interactions per user')
    train = raw[pos < size-2].copy().reset_index(drop=True)
    val = raw[pos == size-2].copy().reset_index(drop=True)
    test = raw[pos == size-1].copy().reset_index(drop=True)
    rng = np.random.default_rng(cfg.split_seed)
    ids = rng.permutation(val.user.to_numpy())
    cut = max(1, min(len(ids)-1, int(len(ids)*cfg.valid_model_fraction)))
    vm = val[val.user.isin(ids[:cut])].sort_values('user').reset_index(drop=True)
    vf = val[val.user.isin(ids[cut:])].sort_values('user').reset_index(drop=True)
    U, I = len(users), len(items)
    ud = np.bincount(train.user, minlength=U).astype(float)
    ideg = np.bincount(train.item, minlength=I).astype(float)
    pairsets = [set(zip(d.user, d.item)) for d in (train, val, test)]
    assert not (pairsets[0]&pairsets[1] or pairsets[0]&pairsets[2] or pairsets[1]&pairsets[2])
    assert set(vm.user).isdisjoint(vf.user)
    assert len(val) == len(test) == U
    train_last = train.groupby('user').timestamp.max()
    assert np.all(train_last.loc[val.user].to_numpy() <= val.timestamp.to_numpy())
    assert np.all(val.timestamp.to_numpy() <= test.timestamp.to_numpy())
    b = dict(name=name, train=train, valid=val, valid_model=vm, valid_fusion=vf,
             test=test, U=U, I=I, user_degree=ud, item_degree=ideg,
             normalization_catalog=np.flatnonzero(ideg > 0), umap=umap, imap=imap)
    manifest = dict(provenance or {}, original_rows=original_n, invalid_removed=original_n-cleaned_n,
        duplicate_pairs_removed=cleaned_n-unique_n, deduplication='latest timestamp; original row breaks ties',
        kcore_user=cfg.min_user, kcore_item=cfg.min_item, kcore_stages=stages,
        final_users=U, final_items=I, final_rows=len(raw), train_rows=len(train),
        valid_rows=len(val), test_rows=len(test), valid_model_users=len(vm), valid_fusion_users=len(vf),
        split='per-user chronological leave-one-out; timestamp ties resolved deterministically',
        future_activity_filtering='k-core uses complete processed graph; not a global causal split',
        cold_item_policy='include processed-catalog items; report zero-TRAIN-degree target counts',
        test_cold_item_targets=int(np.sum(ideg[test.item] == 0)),
        train_hash=array_hash(train[['user','item','rating','timestamp']].to_numpy()),
        valid_hash=array_hash(val[['user','item','rating','timestamp']].to_numpy()),
        test_hash=array_hash(test[['user','item','rating','timestamp']].to_numpy()),
        minimum_timestamp=int(raw.timestamp.min()), maximum_timestamp=int(raw.timestamp.max()))
    b['manifest'] = manifest
    if rec:
        folder = rec.root / name / 'data'; folder.mkdir(parents=True)
        for key in ('train', 'valid', 'valid_model', 'valid_fusion', 'test'):
            b[key][['user','item','rating','timestamp']].to_csv(folder/f'{key}.csv', index=False)
        (folder/'id_mapping.json').write_text(json.dumps(dict(users=users, items=items)))
        (folder/'manifest.json').write_text(json.dumps(manifest, indent=2, default=jsonable))
        rec.add('dataset_statistics', dataset=name, users=U, items=I, train_rows=len(train),
            train_density=len(train)/(U*I), train_sparsity=1-len(train)/(U*I),
            user_degree_cv=float(ud.std()/ud.mean()), item_degree_cv=float(ideg.std()/ideg.mean()),
            user_degree_gini=gini(ud), item_degree_gini=gini(ideg),
            retained_fraction=len(raw)/original_n, test_cold_item_targets=manifest['test_cold_item_targets'])
    return b


DATA_FILE_ALIASES = {
    'health': ('health.csv','ratings_Health_and_Personal_Care.csv','Health_and_Personal_Care.csv'),
    'toys': ('toys.csv','ratings_Toys_and_Games.csv','Toys_and_Games.csv'),
    'videogames': ('VideoGames.csv','ratings_Video_Games.csv','Video_Games.csv'),
}


def input_path(value):
    value=str(value).strip()
    if len(value)>1 and value[0]==value[-1] and value[0] in ('"',"'"):
        value=value[1:-1]
    win=PureWindowsPath(value)
    if os.name!='nt' and len(win.drive)==2 and win.drive[1]==':':
        wsl='/mnt/'+win.drive[0].lower()+'/'+('/'.join(win.parts[1:]))
        raise FileNotFoundError(
            f'Path Windows {value!r} digunakan pada kernel {platform.system()}. '
            f'Jika kernel WSL, periksa {wsl!r}; pada Colab/Linux lain, gunakan lokasi CSV di lingkungan itu.')
    return Path(os.path.expandvars(value)).expanduser()


def data_filename_key(name):
    # Case/underscores/spaces/hyphens are only spelling differences.
    # A .gz suffix is a supported compression wrapper, not a different dataset.
    name=name.casefold()
    if name.endswith('.gz'): name=name[:-3]
    return ''.join(ch for ch in name if ch.isalnum())


def resolve_data_path(name,cfg):
    """Resolve ONLY the configured folder or an explicit file; reject ambiguity."""
    spec=DATASET_SPECS[name]
    explicit=cfg.data_files.get(name)
    if explicit:
        path=input_path(explicit)
        if not path.is_absolute(): path=input_path(cfg.data_dir)/path
        if not path.is_file():
            raise FileNotFoundError(f"CFG.data_files[{name!r}] menunjuk file yang tidak ada: {path.absolute()}")
        return path.resolve()
    folder=input_path(cfg.data_dir)
    if not folder.is_dir():
        raise FileNotFoundError(f'Folder data tidak ditemukan: {folder.absolute()}. Ubah CFG.data_dir ke folder CSV asli.')
    accepted={data_filename_key(v) for v in DATA_FILE_ALIASES[name]}
    files=sorted((p for p in folder.iterdir() if p.is_file()),key=lambda p:p.name.casefold())
    matches=[p for p in files if data_filename_key(p.name) in accepted]
    if len(matches)>1:
        raise ValueError(f"Lebih dari satu file cocok untuk {name}: {', '.join(p.name for p in matches)}. "
                         f"Pilih tepat satu lewat CFG.data_files[{name!r}].")
    if not matches:
        available=[p.name for p in files if p.name.casefold().endswith(('.csv','.csv.gz'))]
        listing=', '.join(available[:20]) or '(tidak ada CSV)'
        raise FileNotFoundError(f"{spec['filename']} atau nama sumber aslinya tidak ditemukan di {folder.resolve()}. "
            f"CSV tersedia: {listing}. Jika namanya berbeda atau berada di subfolder, isi CFG.data_files[{name!r}].")
    return matches[0].resolve()


def check_data_files(config,raise_on_error=True,print_report=True):
    """Check all requested inputs before creating a run or loading large CSVs."""
    rows=[]
    for name in config.datasets:
        try:
            path=resolve_data_path(name,config)
            rows.append(dict(dataset=name,status='FOUND',resolved_path=str(path),bytes=path.stat().st_size,error=''))
        except (FileNotFoundError,PermissionError,ValueError) as e:
            rows.append(dict(dataset=name,status='MISSING_OR_AMBIGUOUS',resolved_path='',bytes=None,error=str(e)))
    frame=pd.DataFrame(rows)
    duplicate=frame.loc[frame.status=='FOUND','resolved_path'].duplicated(keep=False)
    for idx in duplicate[duplicate].index:
        frame.loc[idx,'status']='MISSING_OR_AMBIGUOUS'
        frame.loc[idx,'error']='File yang sama dipetakan ke beberapa kategori. Perbaiki CFG.data_files.'
    if print_report:
        print(f'Kernel: {platform.system()} | Working directory: {Path.cwd()}',flush=True)
        print(f'CFG.data_dir: {config.data_dir}',flush=True)
        print(frame[['dataset','status','resolved_path']].to_string(index=False),flush=True)
        for error in frame.loc[frame.status!='FOUND','error']: print(error,flush=True)
    bad=frame[frame.status!='FOUND']
    if len(bad) and raise_on_error:
        detail='\n'.join(f"- {r.dataset}: {r.error}" for r in bad.itertuples())
        raise FileNotFoundError('Pemeriksaan dataset gagal; training belum dimulai.\n'+detail+
            '\nUbah CFG.data_dir atau CFG.data_files, lalu jalankan check_data_files(CFG). '
            'Mengubah DATA_DIR saja setelah CFG dibuat tidak memperbarui CFG.data_dir.')
    return frame


def load_bundle(name, cfg, rec):
    spec = DATASET_SPECS[name]
    path=resolve_data_path(name,cfg)
    raw = pd.read_csv(path, header=None, names=['c0','c1','rating','timestamp'], dtype={'c0':'string','c1':'string'})
    raw = raw.rename(columns={'c0':'user_raw','c1':'item_raw'} if spec['order']=='user_item'
                             else {'c1':'user_raw','c0':'item_raw'})
    confirmation = cfg.source_confirmations.get(name, {})
    provenance = dict(local_path=str(path.resolve()), raw_sha256=sha_file(path), **spec,
        source_identity_status='author_confirmed' if confirmation.get('verified') else 'UNVERIFIED_HINT',
        source_confirmation=confirmation, raw_count_matches_hint=len(raw)==spec['expected_raw_rows'])
    rec.add('provenance_checklist',dataset=name,category=spec['category'],
        local_path=str(path.resolve()),raw_sha256=provenance['raw_sha256'],raw_rows=len(raw),
        version_hint=spec['version_hint'],hint_source_url=spec['source_url'],
        hint_download_url=spec['download_url'],raw_count_matches_hint=provenance['raw_count_matches_hint'],
        author_confirmed=bool(confirmation.get('verified')),author_record=json.dumps(confirmation),
        note='Counts and filenames are hints, not proof of source identity or access rights.')
    return prepare_frame(raw, name, cfg, rec, provenance)


def gini(x):
    x = np.sort(np.asarray(x, float)); n=len(x)
    return float((2*np.dot(np.arange(1,n+1),x)/(n*x.sum()))-(n+1)/n) if n and x.sum()>0 else 0.


def sparse_values(train, shape, values=None):
    values = np.ones(len(train)) if values is None else values
    return sp.csr_matrix((values, (train.user, train.item)), shape=shape, dtype=np.float64)


class CandidateBank:
    """Shared, saved, nested candidates. Ties use fixed item priorities.

    Requested counts larger than eligibility are explicitly capped. 'full'
    contains every eligible item, not a rejection-sampling approximation.
    """
    def __init__(self, targets, history, n_items, seed, policy='full', n_neg=None):
        seen = defaultdict(set)
        for u,i in zip(history.user, history.item): seen[int(u)].add(int(i))
        self.rows=[]; rng=np.random.default_rng(seed)
        self.tie_priority=np.empty(n_items, int); self.tie_priority[rng.permutation(n_items)]=np.arange(n_items)
        for row in targets.sort_values('user').itertuples(index=False):
            u, pos = int(row.user), int(row.item)
            pool=np.array(sorted(set(range(n_items))-seen[u]-{pos}), dtype=np.int64)
            ur=np.random.default_rng(np.random.SeedSequence([seed,u,pos]))
            perm=pool[ur.permutation(len(pool))]
            negatives=pool if policy=='full' else perm[:min(int(n_neg),len(pool))]
            cand=np.sort(np.r_[pos, negatives])
            self.rows.append((u,pos,cand,len(pool)))
        self.policy=policy; self.requested_negatives=n_neg
        self.digest=array_hash(self.tie_priority, *[np.r_[u,pos,avail,c] for u,pos,c,avail in self.rows])

    def save(self, path):
        arrays={f'items_{u}':c for u,p,c,a in self.rows}
        np.savez_compressed(path, tie_priority=self.tie_priority,
            users=np.array([r[0] for r in self.rows]), targets=np.array([r[1] for r in self.rows]),
            available_negatives=np.array([r[3] for r in self.rows]), **arrays)


@dataclass
class Scores:
    values: np.ndarray
    name: str
    information: str
    rating_values: np.ndarray | None = None
    metadata: dict = field(default_factory=dict)


def evaluate(scores, bank, b, cfg):
    if not np.isfinite(scores.values).all():
        raise FloatingPointError(f'Nonfinite scores from {scores.name}; evaluation aborted')
    detail=[]; exposures={k:np.zeros(b['I']) for k in cfg.ks}
    pop=b['item_degree']; prob=(pop+1)/(pop.sum()+b['I'])
    tail=pop<=np.quantile(pop,1/3)
    for u,pos,cand,available in bank.rows:
        s=scores.values[u,cand]; target_idx=int(np.flatnonzero(cand==pos)[0])
        order=np.lexsort((bank.tie_priority[cand],-s)); rank=int(np.flatnonzero(order==target_idx)[0])+1
        neg=s[np.arange(len(s))!=target_idx]; ps=s[target_idx]
        row=dict(user=u,target=pos,rank=rank,candidate_count=len(cand),
            available_negatives=available,actual_negatives=len(cand)-1,
            requested_negatives=bank.requested_negatives,candidate_hash=bank.digest,
            target_ties=int(np.sum(s==ps)),target_cold=int(pop[pos]==0),
            MRR=1/rank,AUC=float(np.mean((ps>neg)+.5*(ps==neg))) if len(neg) else np.nan,
            user_degree=b['user_degree'][u],target_degree=pop[pos])
        for k in cfg.ks:
            hit=float(rank<=k); top=cand[order[:min(k,len(cand))]]
            row.update({f'HR@{k}':hit,f'Recall@{k}':hit,f'Precision@{k}':hit/k,
                f'NDCG@{k}':1/math.log2(rank+1) if hit else 0.,f'MRR@{k}':1/rank if hit else 0.,
                f'Novelty@{k}':float(np.mean(-np.log2(prob[top]))),
                f'MeanPopularity@{k}':float(np.mean(pop[top])),f'TailShare@{k}':float(np.mean(tail[top]))})
            exposures[k][top]+=1
        detail.append(row)
    detail=pd.DataFrame(detail)
    metric_cols=[c for c in detail if '@' in c]+['MRR','AUC']
    agg={m:float(detail[m].mean()) for m in metric_cols}
    agg.update(users_evaluated=len(detail), MeanRank=float(detail['rank'].mean()),
        TieRate=float((detail.target_ties>1).mean()), CandidateMin=int(detail.candidate_count.min()),
        CandidateMedian=float(detail.candidate_count.median()), CandidateMax=int(detail.candidate_count.max()),
        CappedFraction=float((detail.actual_negatives<bank.requested_negatives).mean()) if bank.policy!='full' else 0.)
    for k in cfg.ks:
        agg[f'CatalogCoverage@{k}']=float(np.mean(exposures[k]>0))
        agg[f'ExposureGini@{k}']=gini(exposures[k])
    return agg,detail


def fixed_normalize(values, catalog):
    ref=values[:,catalog]; mean=ref.mean(axis=1,keepdims=True); sd=ref.std(axis=1,keepdims=True)
    return (values-mean)/np.maximum(sd,1e-12)


def validation_ndcg(scores,bank,k):
    """Exact target rank without sorting or computing unused validation metrics."""
    vals=[]
    for u,pos,cand,_ in bank.rows:
        s=scores.values[u,cand]
        if not np.isfinite(s).all(): raise FloatingPointError('Nonfinite validation score')
        ps=scores.values[u,pos]
        rank=1+int(np.sum(s>ps))+int(np.sum((s==ps)&(bank.tie_priority[cand]<bank.tie_priority[pos])))
        vals.append(1/math.log2(rank+1) if rank<=k else 0.)
    return float(np.mean(vals))


def fit_bias(train,U,I,reg=10.,iterations=8):
    u=train.user.to_numpy(); i=train.item.to_numpy(); r=train.rating.to_numpy(float)
    mu=float(r.mean()); bu=np.zeros(U); bi=np.zeros(I)
    du=np.bincount(u,minlength=U); di=np.bincount(i,minlength=I)
    for _ in range(iterations):
        bu=np.bincount(u,weights=r-mu-bi[i],minlength=U)/(reg+du)
        bi=np.bincount(i,weights=r-mu-bu[u],minlength=I)/(reg+di)
    # Remove the remaining intercept from observed residuals before permutation.
    mu+=float(np.mean(r-mu-bu[u]-bi[i]))
    return mu,bu,bi


def vp_factors(b,power,smooth,clip,axis='both',shuffle_seed=None):
    ud,ideg=b['user_degree'].copy(),b['item_degree'].copy()
    if shuffle_seed is not None:
        rng=np.random.default_rng(shuffle_seed); rng.shuffle(ud); rng.shuffle(ideg)
    uw=((ud[ud>0].mean()+smooth)/(ud+smooth))**(power/2) if axis in ('both','user') else np.ones(len(ud))
    iw=((ideg[ideg>0].mean()+smooth)/(ideg+smooth))**(power/2) if axis in ('both','item') else np.ones(len(ideg))
    # Clip the factors, not entry products: the transform stays separable and
    # inverse diagonal rescaling is exact for the transformed reconstruction.
    for w in (uw,iw):
        lo,hi=np.quantile(w,clip); w[:]=np.clip(w,max(lo,1e-8),max(hi,1e-8))
    t=b['train']; scale=math.sqrt(float(np.mean(uw[t.user]*iw[t.item])))
    return uw/scale,iw/scale


def top_one_verified(A,seed,tol=1e-7):
    A=sp.csr_matrix(A,dtype=float)
    if not A.nnz or np.linalg.norm(A.data)==0: return 0.,0.
    if min(A.shape)<3:
        return float(scipy.linalg.svdvals(A.toarray())[0]),0.
    v0=np.random.default_rng(seed).normal(size=min(A.shape))
    u,s,vt=svds(A,k=1,which='LM',tol=tol,maxiter=10000,v0=v0)
    val=float(s[0]); v=vt[0]; uu=u[:,0]
    residual=max(np.linalg.norm(A@v-val*uu),np.linalg.norm(A.T@uu-val*v))/max(val,1e-15)
    if residual>max(10*tol,1e-6): raise ArithmeticError(f'Spectral norm residual={residual}')
    return val,float(residual)


def observed_svd(A,rank,seed,cfg):
    rank=max(1,min(int(rank),min(A.shape)-1))
    U,s,Vt=randomized_svd(A,n_components=rank,n_iter=8,random_state=seed)
    exact,res=top_one_verified(A,seed+919,cfg.svd_tol)
    rel=abs(float(s[0])-exact)/max(exact,1e-15); fallback=False
    if rel>cfg.svd_verify_tol:
        U,s,Vt=svds(A,k=rank,which='LM',tol=cfg.svd_tol,maxiter=20000,
                     v0=np.random.default_rng(seed).normal(size=min(A.shape)))
        order=np.argsort(-s); U,s,Vt=U[:,order],s[order],Vt[order]; fallback=True
        rel=abs(float(s[0])-exact)/max(exact,1e-15)
    if rel>cfg.svd_verify_tol: raise ArithmeticError('Observed SVD top-value verification failed')
    return U,s,Vt,dict(top_reference=exact,top_relative_error=rel,solver_residual=res,svds_fallback=fallback)


def null_edges(t,residual,uw,iw,shape,count,seed,cfg,kind='residual'):
    rng=np.random.default_rng(seed); u=t.user.to_numpy(); i=t.item.to_numpy(); edges=[]; checks=[]; collisions=[]
    for k in range(count):
        if kind=='residual':
            ii=i; x=residual[rng.permutation(len(residual))]*uw[u]*iw[ii]
        elif kind=='stub':
            ii=i[rng.permutation(len(i))]; x=uw[u]*iw[ii]
        else: raise ValueError(kind)
        H=sp.csr_matrix((x,(u,ii)),shape=shape)
        edge,check=top_one_verified(H,seed+1009*k,cfg.svd_tol)
        edges.append(edge); checks.append(check)
        collisions.append(1-H.nnz/len(t) if kind=='stub' else 0.)
    return np.array(edges),np.array(checks),np.array(collisions)


def shrink(s,edge,mode):
    s=np.asarray(s,float)
    if mode=='none': return s.copy()
    if mode=='hard': return np.where(s>edge,s,0.)
    if mode=='soft': return np.maximum(s-edge,0.)
    if mode=='excess': return np.sqrt(np.maximum(s*s-edge*edge,0.))
    if mode=='reliability': return s*s*s/(s*s+edge*edge+1e-30)
    raise ValueError(mode)


def spectral(b,cfg,seed,params=None,power=None,axis='both',shrinker='reliability',implicit=False,
             edge_override=None,shuffle=False,permutations=None):
    p=dict(rank=cfg.max_rank,scale=1.,bias_reg=cfg.bias_reg); p.update(params or {})
    power=cfg.vp_power if power is None else power
    t=b['train']; U,I=b['U'],b['I']; u=t.user.to_numpy(); i=t.item.to_numpy()
    if implicit: mu,bu,bi=0.,np.zeros(U),np.zeros(I); r=np.ones(len(t)); kind='stub'
    else:
        mu,bu,bi=fit_bias(t,U,I,p['bias_reg']); r=t.rating.to_numpy(float)-mu-bu[u]-bi[i]; kind='residual'
    uw,iw=vp_factors(b,power,cfg.vp_smooth,cfg.factor_clip,axis,seed+151 if shuffle else None)
    X=sp.csr_matrix((r*uw[u]*iw[i],(u,i)),shape=(U,I))
    Uf,s,Vt,check=observed_svd(X,p['rank'],seed,cfg)
    if edge_override is None:
        edges,checks,collisions=null_edges(t,r,uw,iw,(U,I),permutations or cfg.permutations,seed+10001,cfg,kind)
        empirical=float(np.quantile(edges,cfg.edge_quantile))
    else: empirical=float(edge_override); edges=checks=collisions=np.array([])
    used=empirical*p['scale']; sh=shrink(s,used,shrinker)
    latent=((Uf/uw[:,None])*sh)@(Vt/iw[None,:])
    rating=mu+bu[:,None]+bi[None,:]+latent
    vals=latent+bi[None,:]
    entropy=math.exp(-np.sum((sh/sh.sum())*np.log(sh/sh.sum()+1e-30))) if sh.sum() else 0.
    meta=dict(params=p,power=power,axis=axis,shrinker=shrinker,implicit=implicit,
        null_family=kind,empirical_q_edge=empirical,shrinkage_scale=p['scale'],shrinkage_edge=used,
        # Detection is ALWAYS against the unscaled empirical null quantile.
        detectable_rank=int(np.sum(s>empirical)),algebraic_rank=int(np.sum(sh>1e-12)),
        entropy_rank=entropy,retained_energy=float(np.sum(sh*sh)/max(np.sum(s*s),1e-30)),
        raw_singulars=s,shrunk_singulars=sh,null_edges=edges,
        max_null_solver_residual=float(checks.max()) if len(checks) else 0.,
        collision_mean=float(collisions.mean()) if len(collisions) else 0.,
        user_weights=uw,item_weights=iw,**check)
    return Scores(vals,'Spectral','implicit_edges' if implicit else 'ratings_and_edges',
                  None if implicit else rating,meta)


def require_torch():
    if torch is None:
        raise ImportError('PyTorch is required for BPRMF/LightGCN/LightGCL. Install requirements first.')


def exact_negative_batch(rng,users,seen,n_items):
    neg=rng.integers(n_items,size=len(users)); bad=np.array([x in seen[int(u)] for u,x in zip(users,neg)])
    for _ in range(12):
        if not bad.any(): return neg
        neg[bad]=rng.integers(n_items,size=int(bad.sum()))
        bad=np.array([x in seen[int(u)] for u,x in zip(users,neg)])
    for idx in np.flatnonzero(bad):
        pool=np.array(sorted(set(range(n_items))-seen[int(users[idx])]))
        if not len(pool): raise ValueError('User has no available training negatives')
        neg[idx]=rng.choice(pool)
    assert all(x not in seen[int(u)] for u,x in zip(users,neg))
    return neg


def torch_device(cfg):
    require_torch()
    return ('cuda' if torch.cuda.is_available() else 'cpu') if cfg.device=='auto' else cfg.device


def normalized_graph(b,device):
    t=b['train']; u=t.user.to_numpy(); i=t.item.to_numpy()
    w=1/np.sqrt(np.maximum(b['user_degree'][u],1)*np.maximum(b['item_degree'][i],1))
    idx=torch.tensor(np.vstack([u,i]),dtype=torch.long,device=device)
    rect=torch.sparse_coo_tensor(idx,torch.tensor(w,dtype=torch.float32,device=device),
        size=(b['U'],b['I']),check_invariants=True).coalesce()
    inode=b['U']+i
    idx2=torch.tensor(np.vstack([np.r_[u,inode],np.r_[inode,u]]),dtype=torch.long,device=device)
    square=torch.sparse_coo_tensor(idx2,torch.tensor(np.r_[w,w],dtype=torch.float32,device=device),
        size=(b['U']+b['I'],b['U']+b['I']),check_invariants=True).coalesce()
    return rect,square,sp.csr_matrix((w,(u,i)),shape=(b['U'],b['I']))


def full_logsumexp(anchors,all_nodes,temp,chunk=512):
    # Exact denominator; chunking neither deduplicates nor samples anchors.
    values=[]
    for st in range(0,len(anchors),chunk):
        a=anchors[st:st+chunk]
        values.append(torch.logsumexp(a@all_nodes.T/temp,dim=1))
    return torch.cat(values)


def train_neural(name,b,cfg,seed,p,bank,rec=None,phase='evidence',trial=None):
    require_torch(); seed_all(seed); dev=torch_device(cfg)
    U,I=b['U'],b['I']; dim=cfg.dim
    user=torch.empty(U,dim,device=dev); item=torch.empty(I,dim,device=dev)
    if name=='LightGCL': nn.init.xavier_uniform_(user); nn.init.xavier_uniform_(item)
    else: nn.init.normal_(user,std=.02); nn.init.normal_(item,std=.02)
    user=nn.Parameter(user); item=nn.Parameter(item)
    ib=nn.Parameter(torch.zeros(I,device=dev)) if name=='BPRMF' else None
    params=[user,item]+([ib] if ib is not None else [])
    opt=torch.optim.Adam(params,lr=p.get('lr',.001))
    rect=square=None; svd_data=None
    if name!='BPRMF':
        rect,square,A=normalized_graph(b,dev)
        if name=='LightGCL':
            q=min(p.get('q',5),min(U,I)-1)
            su,ss,sv=randomized_svd(A,n_components=q,n_iter=8,random_state=seed)
            svd_data=[torch.tensor(x,dtype=torch.float32,device=dev) for x in
                      (su*ss,sv.T*ss,su.T,sv)]

    def propagate():
        if name=='BPRMF': return user,item,None,None
        if name=='LightGCN':
            x=torch.cat([user,item]); layers=[x]
            for _ in range(p['layers']): x=torch.sparse.mm(square,x); layers.append(x)
            x=torch.stack(layers).mean(0)
            return x[:U],x[U:],None,None
        eu,ei=user,item; eus=[eu]; eis=[ei]; gus=[eu]; gis=[ei]
        us,vs,ut,vt=svd_data
        for _ in range(p['layers']):
            gus.append(us@(vt@ei)); gis.append(vs@(ut@eu))
            eu,ei=torch.sparse.mm(rect,ei),torch.sparse.mm(rect.transpose(0,1),eu)
            eus.append(eu); eis.append(ei)
        return torch.stack(eus).sum(0),torch.stack(eis).sum(0),torch.stack(gus).sum(0),torch.stack(gis).sum(0)

    def snapshot(epoch):
        with torch.no_grad():
            uf,it,_,_=propagate(); vals=(uf@it.T).cpu().numpy().copy()
            if ib is not None: vals+=ib.detach().cpu().numpy()[None,:]
        return Scores(vals,name,'implicit_edges',metadata=dict(params=p,best_epoch=epoch))

    t=b['train']; ua=t.user.to_numpy(); ia=t.item.to_numpy(); N=len(t)
    seen=defaultdict(set)
    for u,i in zip(ua,ia): seen[int(u)].add(int(i))
    rng=np.random.default_rng(seed); best=-np.inf; best_state=None; best_epoch=0; bad=0
    eval_seconds=0.; calls=0; start=time.perf_counter(); hit_cap=False
    for epoch in range(1,cfg.max_epochs+1):
        losses=[]; idx=rng.permutation(N)
        for st in range(0,N,cfg.batch):
            ix=idx[st:st+cfg.batch]; ub=ua[ix]; pi=ia[ix]
            ni=exact_negative_batch(rng,ub,seen,I)
            tu=torch.tensor(ub,dtype=torch.long,device=dev)
            tp=torch.tensor(pi,dtype=torch.long,device=dev); tn=torch.tensor(ni,dtype=torch.long,device=dev)
            eu,ei,gu,gi=propagate()
            spv=(eu[tu]*ei[tp]).sum(1); snv=(eu[tu]*ei[tn]).sum(1)
            if ib is not None: spv+=ib[tp]; snv+=ib[tn]
            loss=-torch.nn.functional.logsigmoid(spv-snv).mean()
            if name=='LightGCL':
                # Matches official batch multiplicity, aggregation, unclipped
                # embeddings, and positive clamp. No unique-anchor subsampling.
                ii=torch.cat([tp,tn]); temp=p['temp']
                pos=torch.clamp((gu[tu]*eu[tu]).sum(1)/temp,-5.,5.).mean()
                pos+=torch.clamp((gi[ii]*ei[ii]).sum(1)/temp,-5.,5.).mean()
                neg=full_logsumexp(gu[tu],eu,temp).mean()+full_logsumexp(gi[ii],ei,temp).mean()
                loss+=p['lambda1']*(neg-pos)
                if p.get('reg',0)>0: loss+=p['reg']*(user.square().sum()+item.square().sum())
            else:
                reg=(user[tu].square().sum()+item[tp].square().sum()+item[tn].square().sum())/(2*len(ix))
                if ib is not None: reg+=(ib[tp].square().sum()+ib[tn].square().sum())/(2*len(ix))
                loss+=p.get('reg',1e-5)*reg
            if not torch.isfinite(loss): raise FloatingPointError(f'{name} training loss is nonfinite')
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); losses.append(float(loss.detach().cpu()))
        if rec:
            rec.add('training_log',dataset=b['name'],model=name,seed=seed,phase=phase,trial=trial,
                epoch=epoch,train_loss=float(np.mean(losses)),positive_updates=N,coverage=1.)
        if epoch%cfg.eval_every==0 or epoch==cfg.max_epochs:
            ts=time.perf_counter(); m=snapshot(epoch)
            obj=validation_ndcg(m,bank,cfg.primary_k); eval_seconds+=time.perf_counter()-ts; calls+=1
            if rec: rec.add('validation_training',dataset=b['name'],model=name,seed=seed,
                phase=phase,trial=trial,epoch=epoch,validation_ndcg=obj,candidate_hash=bank.digest)
            if obj>best+1e-12:
                best=obj; best_epoch=epoch; bad=0
                best_state=[v.detach().clone() for v in params]
            else: bad+=1
            if bad>=cfg.patience: break
    hit_cap=epoch==cfg.max_epochs and bad<cfg.patience
    with torch.no_grad():
        if best_state is not None:
            for dst,src in zip(params,best_state): dst.copy_(src)
    m=snapshot(best_epoch or epoch)
    m.metadata.update(validation_ndcg=best,epochs_run=epoch,positive_updates=N*epoch,
        validation_calls=calls,early_stopping_seconds=eval_seconds,
        fit_seconds=time.perf_counter()-start,hit_epoch_cap=hit_cap,
        baseline_reference='https://github.com/HKUDS/LightGCL' if name=='LightGCL' else
                           'https://github.com/gusye1234/LightGCN-PyTorch' if name=='LightGCN' else 'BPR with item bias')
    return m


def fit_ials(b,cfg,seed,p,bank,rec=None,phase='evidence',trial=None):
    A=sparse_values(b['train'],(b['U'],b['I'])); At=A.T.tocsr()
    rng=np.random.default_rng(seed); X=rng.normal(0,.02,(b['U'],cfg.dim)); Y=rng.normal(0,.02,(b['I'],cfg.dim))
    reg=p['reg']; alpha=p['alpha']; best=-np.inf; best_pair=None; bad=0; calls=0
    for epoch in range(1,cfg.max_epochs+1):
        for dst,src,mat in ((X,Y,A),(Y,X,At)):
            gram=src.T@src+reg*np.eye(cfg.dim)
            for u in range(mat.shape[0]):
                ix=mat.indices[mat.indptr[u]:mat.indptr[u+1]]; f=src[ix]
                dst[u]=scipy.linalg.solve(gram+alpha*(f.T@f),(1+alpha)*f.sum(0),assume_a='pos')
        if epoch%cfg.eval_every==0 or epoch==cfg.max_epochs:
            m=Scores(X@Y.T,'iALS','implicit_edges'); obj=validation_ndcg(m,bank,cfg.primary_k); calls+=1
            if rec: rec.add('validation_training',dataset=b['name'],model='iALS',seed=seed,
                phase=phase,trial=trial,epoch=epoch,validation_ndcg=obj,candidate_hash=bank.digest)
            if obj>best+1e-12: best=obj; best_pair=(X.copy(),Y.copy(),epoch); bad=0
            else: bad+=1
            if bad>=cfg.patience: break
    xx,yy,ep=best_pair
    return Scores(xx@yy.T,'iALS','implicit_edges',metadata=dict(params=p,best_epoch=ep,
        epochs_run=epoch,validation_ndcg=best,validation_calls=calls,hit_epoch_cap=epoch==cfg.max_epochs))


def fit_base(name,b,cfg,seed,p,bank,rec=None,phase='evidence',trial=None):
    if name in ('BPRMF','LightGCN','LightGCL'):
        return train_neural(name,b,cfg,seed,p,bank,rec,phase,trial)
    if name=='iALS': return fit_ials(b,cfg,seed,p,bank,rec,phase,trial)
    if name=='Popularity':
        return Scores(np.broadcast_to(b['item_degree'],(b['U'],b['I'])).copy(),name,'implicit_edges')
    if name=='EASE':
        A=sparse_values(b['train'],(b['U'],b['I'])); G=(A.T@A).toarray()
        G.flat[::len(G)+1]+=p['reg']; P=scipy.linalg.solve(G,np.eye(b['I']),assume_a='pos')
        B=-P/np.diag(P)[None,:]; np.fill_diagonal(B,0.)
        return Scores(A@B,name,'implicit_edges',metadata=dict(params=p))
    if name in ('PureSVD_Implicit','PureSVD_Ratings'):
        explicit=name=='PureSVD_Ratings'
        A=sparse_values(b['train'],(b['U'],b['I']),b['train'].rating.to_numpy() if explicit else None)
        U,s,Vt,check=observed_svd(A,p['rank'],seed,cfg); values=(U*s)@Vt
        return Scores(values,name,'ratings_and_edges' if explicit else 'implicit_edges',
                      values.copy() if explicit else None,dict(params=p,**check))
    raise ValueError(name)


def configs_for(name,cfg):
    if name=='Popularity': return [{}]
    if name=='BPRMF': grid=[dict(lr=lr,reg=reg) for lr,reg in itertools.product((.0005,.001,.005),(1e-6,1e-5,1e-4,1e-3))]
    elif name=='LightGCN': grid=[dict(layers=l,lr=lr,reg=reg) for l,lr,reg in itertools.product((1,2,3),(.001,.005),(1e-5,1e-3))]
    elif name=='LightGCL': grid=[dict(layers=l,lambda1=w,temp=temp,lr=.001,reg=0.,q=5) for l,w,temp in itertools.product((1,2),(.01,.1,.2),(.1,.2))]
    elif name=='iALS': grid=[dict(alpha=a,reg=r) for a,r in itertools.product((5.,20.,50.),(.01,.1,1.,10.))]
    elif name=='EASE': grid=[dict(reg=float(r)) for r in np.geomspace(1,10000,12)]
    elif name.startswith('PureSVD'):
        grid=[dict(rank=int(r)) for r in np.linspace(max(2,cfg.max_rank//4),max(4,cfg.max_rank*2),12)]
    elif name=='Spectral_Implicit':
        # Bias regularization has no role for binary input: do not waste trials
        # on duplicate ineffective configurations.
        grid=[dict(rank=r,scale=s,bias_reg=10.) for r,s in itertools.product(
            (max(2,cfg.max_rank//4),max(3,cfg.max_rank//2),cfg.max_rank),(.5,.75,1.,1.5))]
    elif name=='Spectral':
        grid=[dict(rank=r,scale=s,bias_reg=br) for r,s,br in itertools.product(
            (max(2,cfg.max_rank//2),cfg.max_rank),(.5,1.,1.5),(10.,50.))]
    else: raise ValueError(name)
    # Evenly spread a reduced smoke budget; paper uses all 12 listed trials.
    indices=np.linspace(0,len(grid)-1,min(cfg.n_trials,len(grid))).round().astype(int)
    return [grid[i] for i in indices]


def tune_base(name,b,cfg,bank,rec):
    best=-np.inf; chosen=None; start=time.perf_counter(); failed=0; calls=0
    for trial,p in enumerate(configs_for(name,cfg)):
        st=time.perf_counter()
        try:
            m=fit_base(name,b,cfg,cfg.tuning_seed,p,bank,rec,'tuning',trial)
            obj=validation_ndcg(m,bank,cfg.primary_k); calls+=1+m.metadata.get('validation_calls',0)
            rec.add('tuning_trials',dataset=b['name'],model=name,trial=trial,params=json.dumps(p),
                objective=obj,seconds=time.perf_counter()-st,status='ok',candidate_hash=bank.digest,
                hit_epoch_cap=m.metadata.get('hit_epoch_cap',False))
            if obj>best+1e-12: best=obj; chosen=p
        except Exception as e:
            failed+=1; rec.add('tuning_trials',dataset=b['name'],model=name,trial=trial,params=json.dumps(p),
                status='failed',error=repr(e),seconds=time.perf_counter()-st)
            # A partial baseline search is never silently treated as complete.
            raise RuntimeError(f'{name} tuning trial {trial} failed') from e
    rec.add('tuning_budget',dataset=b['name'],model=name,trials=len(configs_for(name,cfg)),
        failed_trials=failed,validation_calls=calls,tuning_seconds=time.perf_counter()-start,
        selected_params=json.dumps(chosen),selection_metric=f'NDCG@{cfg.primary_k}',
        validation_partition='valid_model',tuning_seed=cfg.tuning_seed)
    return chosen


def tune_spectral(b,cfg,bank,rec,implicit=False):
    best=-np.inf; chosen=None; start=time.perf_counter(); cache={}; name='Spectral_Implicit' if implicit else 'Spectral_Explicit'
    grid=configs_for('Spectral_Implicit' if implicit else 'Spectral',cfg)
    for trial,p in enumerate(grid):
        # Reuse identical raw decomposition/null only within this tuning job;
        # total tuning time is still charged to this pipeline.
        key=(p['rank'],p['bias_reg'])
        st=time.perf_counter()
        if key not in cache:
            q=dict(p,scale=1.)
            cache[key]=spectral(b,cfg,cfg.tuning_seed,q,implicit=implicit)
        base=cache[key]
        m=spectral(b,cfg,cfg.tuning_seed,p,implicit=implicit,
                   edge_override=base.metadata['empirical_q_edge'])
        obj=validation_ndcg(m,bank,cfg.primary_k)
        rec.add('tuning_trials',dataset=b['name'],model=name,trial=trial,params=json.dumps(p),
            objective=obj,seconds=time.perf_counter()-st,status='ok',candidate_hash=bank.digest)
        if obj>best+1e-12: best=obj; chosen=p
    rec.add('tuning_budget',dataset=b['name'],model=name,trials=len(grid),
        validation_calls=len(grid),tuning_seconds=time.perf_counter()-start,
        selected_params=json.dumps(chosen),selection_metric=f'NDCG@{cfg.primary_k}',
        validation_partition='valid_model',tuning_seed=cfg.tuning_seed)
    return chosen


@dataclass
class FusionSearch:
    bpr: Scores
    spectral: Scores
    bpr_z: np.ndarray
    spectral_z: np.ndarray
    popularity_z: np.ndarray
    baseline_objective: float
    selected_objective: float
    pre_gate_spectral_weight: float
    pre_gate_popularity_weight: float
    search_calls: int


def search_fusion(base,spec,b,bank,cfg,rec=None,seed=None,model_name='VP_RMT_SVD',
                  spectral_allowed=True,popularity_allowed=True):
    bz=fixed_normalize(base.values,b['normalization_catalog'])
    sz=fixed_normalize(spec.values,b['normalization_catalog'])
    pop=np.broadcast_to(np.log1p(b['item_degree']),(b['U'],b['I'])).copy()
    pz=fixed_normalize(pop,b['normalization_catalog'])
    base_obj=validation_ndcg(base,bank,cfg.primary_k)
    best=base_obj; sw=pw=0.; calls=1
    sg=cfg.fusion_grid if spectral_allowed else (0.,); pg=cfg.fusion_grid if popularity_allowed else (0.,)
    # Ties prefer the exact backbone then the smaller absolute coefficients.
    pairs=sorted(itertools.product(sg,pg),key=lambda x:(abs(x[0])+abs(x[1]),x))
    for a,c in pairs:
        values=base.values if a==c==0 else bz+a*sz+c*pz
        obj=validation_ndcg(Scores(values,model_name,spec.information),bank,cfg.primary_k); calls+=1
        if rec: rec.add('fusion_trials',dataset=b['name'],seed=seed,model=model_name,
            spectral_weight=a,popularity_weight=c,validation_ndcg=obj,candidate_hash=bank.digest,
            partition='valid_fusion')
        if obj>best+1e-12: best=obj; sw=float(a); pw=float(c)
    return FusionSearch(base,spec,bz,sz,pz,base_obj,best,sw,pw,calls)


def deploy_fusion(search,delta,name='VP_RMT_SVD'):
    gain=search.selected_objective-search.baseline_objective
    rejected=gain<float(delta)
    sw=0. if rejected else search.pre_gate_spectral_weight
    pw=0. if rejected else search.pre_gate_popularity_weight
    values=search.bpr.values.copy() if sw==pw==0 else search.bpr_z+sw*search.spectral_z+pw*search.popularity_z
    meta=dict(gate_delta=delta,pre_gate_gain=gain,gate_rejected=rejected,
        pre_gate_spectral_weight=search.pre_gate_spectral_weight,
        pre_gate_popularity_weight=search.pre_gate_popularity_weight,
        spectral_weight=sw,popularity_weight=pw,exact_backbone_fallback=sw==pw==0,
        validation_calls=search.search_calls,baseline_objective=search.baseline_objective,
        pre_gate_objective=search.selected_objective)
    return Scores(values,name,search.spectral.information,
                  search.spectral.rating_values,meta)


def rating_metrics(m,b):
    if m.rating_values is None: return dict(RMSE=np.nan,MAE=np.nan)
    # Same two-parameter validation calibration for every rating-capable model.
    v=b['valid_model']; raw=m.rating_values[v.user,v.item]; y=v.rating.to_numpy(float)
    X=np.c_[raw,np.ones(len(raw))]; coef=np.linalg.lstsq(X,y,rcond=None)[0]
    tt=b['test']; pred=coef[0]*m.rating_values[tt.user,tt.item]+coef[1]
    pred=np.clip(pred,b['train'].rating.min(),b['train'].rating.max()); e=pred-tt.rating.to_numpy(float)
    return dict(RMSE=float(np.sqrt(np.mean(e*e))),MAE=float(np.mean(np.abs(e))),
                rating_calibration_a=coef[0],rating_calibration_b=coef[1])


BASE_MODELS=('Popularity','PureSVD_Implicit','PureSVD_Ratings','BPRMF','LightGCN','LightGCL','iALS','EASE')
MAIN_MODELS=BASE_MODELS+('VP_RMT_SVD','VP_RMT_Implicit','LightGCN_MatchedFusion')
CONTRASTS=(
    ('VP_RMT_SVD','BPRMF','overall_vs_backbone'),
    ('VP_RMT_SVD','LightGCN','explicit_vs_implicit_reference'),
    ('VP_RMT_SVD','LightGCL','explicit_vs_implicit_reference'),
    ('VP_RMT_SVD','PureSVD_Ratings','explicit_reference'),
    ('VP_RMT_SVD','LightGCN_MatchedFusion','matched_information_and_fusion'),
    ('VP_RMT_Implicit','LightGCN','implicit_information_parity'),
    ('VP_RMT_Implicit','LightGCL','implicit_information_parity'),
    ('VP_RMT_Implicit','iALS','implicit_information_parity'),
    ('VP_RMT_Implicit','EASE','implicit_information_parity'),
    ('VP_RMT_SVD','VP_NoGate','margin_gate_only'),
    ('VP_RMT_SVD','VP_NoVP','matched_weighting'),
    ('VP_RMT_SVD','VP_NoShrink','matched_shrinkage'),
    ('VP_RMT_SVD','VP_Hard','matched_shrinkage'),
    ('VP_RMT_SVD','VP_Soft','matched_shrinkage'),
    ('VP_RMT_SVD','VP_Excess','matched_shrinkage'),
    ('VP_RMT_SVD','BPR_Pop','spectral_beyond_popularity'),
    ('VP_RMT_SVD','BPR_Spectral','popularity_beyond_spectral'),
    ('VP_RMT_SVD','VP_UserOnly','weighting_axis'),
    ('VP_RMT_SVD','VP_ItemOnly','weighting_axis'),
    ('VP_RMT_SVD','VP_ShuffledDegree','degree_alignment_control'),
    ('VP_RMT_SVD_Spectral','VP_NoVP_Spectral','spectral_only_weighting'),
    ('VP_RMT_SVD_Spectral','VP_NoShrink_Spectral','spectral_only_shrinkage'),
    ('VP_RMT_SVD_Spectral','VP_Hard_Spectral','spectral_only_shrinkage'),
    ('VP_RMT_SVD_Spectral','VP_Soft_Spectral','spectral_only_shrinkage'),
    ('VP_RMT_SVD_Spectral','VP_Excess_Spectral','spectral_only_shrinkage'),
)


def record_evaluation(m,b,bank,cfg,rec,seed,stage='main',extra=None):
    start=time.perf_counter(); agg,detail=evaluate(m,bank,b,cfg)
    elapsed=time.perf_counter()-start
    detail['dataset']=b['name']; detail['seed']=seed; detail['model']=m.name
    detail['policy']=bank.policy; detail['stage']=stage
    folder=rec.root/b['name']/'details'; folder.mkdir(parents=True,exist_ok=True)
    suffix=f"{stage}_{bank.policy}_{bank.requested_negatives}_{(extra or {}).get('candidate_rep',0)}"
    detail.to_csv(folder/f'{m.name}_seed{seed}_{suffix}.csv',index=False)
    row=dict(dataset=b['name'],seed=seed,model=m.name,information=m.information,
        stage=stage,policy=bank.policy,requested_negatives=bank.requested_negatives,
        candidate_hash=bank.digest,ranking_evaluation_seconds=elapsed,
        score_matrix_bytes=int(m.values.nbytes),
        evaluation_cost_scope='Scores generated in fit; time here measures cached-score ranking and metrics only',
        **agg,**(extra or {}))
    if stage=='main': row.update(rating_metrics(m,b))
    rec.add('metrics',**row)
    if stage=='main':
        rec.add('detail_index',dataset=b['name'],seed=seed,model=m.name,
            path=str((folder/f'{m.name}_seed{seed}_{suffix}.csv').relative_to(rec.root)),candidate_hash=bank.digest)
        np.save(folder/f'{m.name}_seed{seed}_scores.npy',m.values.astype(np.float32))
        for axis,col in (('user_activity','user_degree'),('target_popularity','target_degree')):
            pct=detail[col].rank(method='average',pct=True)
            groups=np.where(pct<=1/3,'low',np.where(pct<=2/3,'mid','high'))
            for label in ('low','mid','high'):
                dd=detail[groups==label]
                rec.add('degree_regime_metrics',dataset=b['name'],seed=seed,model=m.name,
                    axis=axis,group=label,n_users=len(dd),mean_degree=dd[col].mean(),
                    ndcg=dd[f'NDCG@{cfg.primary_k}'].mean(),hr=dd[f'HR@{cfg.primary_k}'].mean())
    return row


def log_spectral(m,b,rec,seed,name):
    d=m.metadata
    rec.add('spectral_diagnostics',dataset=b['name'],seed=seed,model=name,
        **{k:v for k,v in d.items() if not isinstance(v,(np.ndarray,dict))})
    for j,(s,h) in enumerate(zip(d['raw_singulars'],d['shrunk_singulars'])):
        rec.add('singular_values',dataset=b['name'],seed=seed,model=name,index=j+1,
            raw=s,shrunk=h,empirical_q_edge=d['empirical_q_edge'],shrinkage_edge=d['shrinkage_edge'])
    if len(d['null_edges']):
        for j,x in enumerate(d['null_edges']):
            rec.add('permutation_edges',dataset=b['name'],seed=seed,model=name,permutation=j,value=x)


def degree_diagnostics(b,cfg,rec):
    t=b['train']; u=t.user.to_numpy(); i=t.item.to_numpy()
    mu,bu,bi=fit_bias(t,b['U'],b['I'],cfg.bias_reg)
    residual=t.rating.to_numpy(float)-mu-bu[u]-bi[i]
    uw,iw=vp_factors(b,cfg.vp_power,cfg.vp_smooth,cfg.factor_clip)
    weighted=residual*uw[u]*iw[i]
    for axis,ids,deg in (('user',u,b['user_degree']),('item',i,b['item_degree'])):
        pct=rankdata(deg,method='average')/len(deg); labels=np.digitize(pct[ids],[1/3,2/3])
        before=[]; after=[]
        for k in range(3):
            mask=labels==k
            vb=np.var(residual[mask],ddof=1) if mask.sum()>1 else np.nan
            va=np.var(weighted[mask],ddof=1) if mask.sum()>1 else np.nan
            before.append(vb); after.append(va)
            rec.add('residual_degree_bins',dataset=b['name'],axis=axis,group=k,n=int(mask.sum()),
                residual_variance=vb,weighted_residual_variance=va)
        cv=lambda x:float(np.nanstd(x)/max(np.nanmean(x),1e-15))
        # Row/column energy includes the exact number of observations and is a
        # different diagnostic from conditional residual variance.
        en=np.bincount(ids,weights=residual**2,minlength=len(deg))
        ew=np.bincount(ids,weights=weighted**2,minlength=len(deg))
        rec.add('degree_diagnostics',dataset=b['name'],axis=axis,
            conditional_variance_cv_before=cv(before),conditional_variance_cv_after=cv(after),
            matrix_energy_cv_before=float(en.std()/max(en.mean(),1e-15)),
            matrix_energy_cv_after=float(ew.std()/max(ew.mean(),1e-15)),
            interpretation='Conditional residual variance and sparse-matrix energy are distinct; lower is not guaranteed.')


def run_dataset(b,cfg,rec):
    name=b['name']; folder=rec.root/name; folder.mkdir(exist_ok=True)
    banks={}
    for key in ('valid_model','valid_fusion','test'):
        history=b['train'] if key!='test' else pd.concat([b['train'],b['valid']],ignore_index=True)
        bank=CandidateBank(b[key],history,b['I'],cfg.candidate_seed,policy='full')
        bank.save(folder/f'{key}_full_candidates.npz'); banks[key]=bank
        rec.add('candidate_manifest',dataset=name,partition=key,policy='full',hash=bank.digest,
            users=len(bank.rows),min_candidates=min(len(x[2]) for x in bank.rows),
            max_candidates=max(len(x[2]) for x in bank.rows))
    print(f'[{name}] tuning on {len(banks["valid_model"].rows)} users; fusion on {len(banks["valid_fusion"].rows)} other users',flush=True)
    selected={m:tune_base(m,b,cfg,banks['valid_model'],rec) for m in BASE_MODELS}
    sp_exp=tune_spectral(b,cfg,banks['valid_model'],rec,False)
    sp_imp=tune_spectral(b,cfg,banks['valid_model'],rec,True)
    rec.dump_json(f'{name}_selected_parameters.json',dict(baselines=selected,spectral_explicit=sp_exp,spectral_implicit=sp_imp))
    degree_diagnostics(b,cfg,rec)
    anchor_state=None
    for seed in cfg.seeds:
        print(f'[{name}] evidence seed {seed}',flush=True)
        fitted={}; times={}
        for model in BASE_MODELS:
            st=time.perf_counter()
            m=fit_base(model,b,cfg,seed,selected[model],banks['valid_model'],rec)
            times[model]=time.perf_counter()-st; fitted[model]=m
            rec.add('runtime',dataset=name,seed=seed,model=model,kind='standalone_cold_fit',
                seconds=times[model],includes_validation_selection=True,cache_used=False,
                epochs_run=m.metadata.get('epochs_run'),hit_epoch_cap=m.metadata.get('hit_epoch_cap',False))
            record_evaluation(m,b,banks['test'],cfg,rec,seed)
        backbone=fitted['BPRMF']
        # Actual independent cold-fit probes charge the full pipeline. Ablation
        # predictions retain the EXACT baseline backbone object for matching.
        for implicit,main,p in ((False,'VP_RMT_SVD',sp_exp),(True,'VP_RMT_Implicit',sp_imp)):
            cold_start=time.perf_counter()
            probe=fit_base('BPRMF',b,cfg,seed,selected['BPRMF'],banks['valid_model'])
            maxdiff=float(np.max(np.abs(probe.values-backbone.values)))
            ts=time.perf_counter(); spec=spectral(b,cfg,seed,p,implicit=implicit)
            spectral_seconds=time.perf_counter()-ts
            ts=time.perf_counter(); fs=search_fusion(backbone,spec,b,banks['valid_fusion'],cfg,rec,seed,main)
            fusion_seconds=time.perf_counter()-ts
            m=deploy_fusion(fs,cfg.gate_delta,main); fitted[main]=m
            cold_seconds=time.perf_counter()-cold_start
            rec.add('runtime',dataset=name,seed=seed,model=main,kind='standalone_cold_pipeline_probe',
                seconds=cold_seconds,spectral_seconds=spectral_seconds,fusion_seconds=fusion_seconds,
                includes_validation_selection=True,cache_used=False,backbone_repeat_max_difference=maxdiff)
            rec.add('model_state',dataset=name,seed=seed,model=main,**m.metadata)
            log_spectral(spec,b,rec,seed,main+'_Spectral')
            record_evaluation(m,b,banks['test'],cfg,rec,seed)
            spectral_only=copy.copy(spec); spectral_only.name=main+'_Spectral'
            record_evaluation(spectral_only,b,banks['test'],cfg,rec,seed)
            if not implicit:
                full_spec=spec; full_search=fs
                ts=time.perf_counter(); no_gate=deploy_fusion(fs,0.,'VP_NoGate')
                rec.add('runtime',dataset=name,seed=seed,model=no_gate.name,kind='matched_incremental_gate',
                    seconds=time.perf_counter()-ts,cache_used=True,includes_validation_selection=False)
                rec.add('model_state',dataset=name,seed=seed,model=no_gate.name,**no_gate.metadata)
                record_evaluation(no_gate,b,banks['test'],cfg,rec,seed)
                for label,sa,pa in (('BPR_Pop',False,True),('BPR_Spectral',True,False)):
                    ts=time.perf_counter()
                    ff=search_fusion(backbone,spec,b,banks['valid_fusion'],cfg,rec,seed,label,sa,pa)
                    mm=deploy_fusion(ff,cfg.gate_delta,label)
                    rec.add('runtime',dataset=name,seed=seed,model=label,kind='matched_incremental_fusion',
                        seconds=time.perf_counter()-ts,cache_used=True,includes_validation_selection=True)
                    if not sa: mm.information='implicit_edges'; mm.rating_values=None
                    rec.add('model_state',dataset=name,seed=seed,model=label,**mm.metadata)
                    record_evaluation(mm,b,banks['test'],cfg,rec,seed)
        # Every mechanistic contrast uses the selected main rank/scale/bias,
        # same residual/null RNG, same backbone, and same fusion grid/partition.
        variants=[('VP_NoVP',dict(power=0.)),('VP_UserOnly',dict(axis='user')),
            ('VP_ItemOnly',dict(axis='item')),('VP_ShuffledDegree',dict(shuffle=True)),
            ('VP_NoShrink',dict(shrinker='none',edge_override=full_spec.metadata['empirical_q_edge'])),
            ('VP_Hard',dict(shrinker='hard',edge_override=full_spec.metadata['empirical_q_edge'])),
            ('VP_Soft',dict(shrinker='soft',edge_override=full_spec.metadata['empirical_q_edge'])),
            ('VP_Excess',dict(shrinker='excess',edge_override=full_spec.metadata['empirical_q_edge']))]
        for label,kwargs in variants:
            ts=time.perf_counter(); sm=spectral(b,cfg,seed,sp_exp,**kwargs)
            ff=search_fusion(backbone,sm,b,banks['valid_fusion'],cfg,rec,seed,label)
            mm=deploy_fusion(ff,cfg.gate_delta,label)
            rec.add('runtime',dataset=name,seed=seed,model=label,kind='matched_incremental_fit',
                seconds=time.perf_counter()-ts,cache_used=True,includes_validation_selection=True)
            rec.add('model_state',dataset=name,seed=seed,model=label,**mm.metadata)
            log_spectral(sm,b,rec,seed,label+'_Spectral')
            record_evaluation(mm,b,banks['test'],cfg,rec,seed)
            sm.name=label+'_Spectral'; record_evaluation(sm,b,banks['test'],cfg,rec,seed)
        # Vary ONLY the backbone: identical fitted VP spectrum and fusion grid.
        cold_start=time.perf_counter()
        probe=fit_base('LightGCN',b,cfg,seed,selected['LightGCN'],banks['valid_model'])
        maxdiff=float(np.max(np.abs(probe.values-fitted['LightGCN'].values)))
        cold_spec=spectral(b,cfg,seed,sp_exp)
        matched=search_fusion(fitted['LightGCN'],full_spec,b,banks['valid_fusion'],
                             cfg,rec,seed,'LightGCN_MatchedFusion')
        mm=deploy_fusion(matched,cfg.gate_delta,'LightGCN_MatchedFusion'); fitted[mm.name]=mm
        rec.add('runtime',dataset=name,seed=seed,model=mm.name,kind='standalone_cold_pipeline_probe',
            seconds=time.perf_counter()-cold_start,cache_used=False,includes_validation_selection=True,
            backbone_repeat_max_difference=maxdiff)
        rec.add('model_state',dataset=name,seed=seed,model=mm.name,**mm.metadata)
        del probe,cold_spec
        record_evaluation(mm,b,banks['test'],cfg,rec,seed)
        if cfg.run_sensitivities:
            for delta in cfg.gate_grid:
                mm=deploy_fusion(full_search,delta,'GateSensitivity')
                agg,_=evaluate(mm,banks['test'],b,cfg)
                rec.add('gate_sensitivity',dataset=name,seed=seed,**mm.metadata,**agg)
            for policy_rep in range(cfg.candidate_repeats):
                candidate_seed=cfg.candidate_seed+policy_rep*100003
                for n in cfg.negative_grid:
                    bank=CandidateBank(b['test'],pd.concat([b['train'],b['valid']]),b['I'],candidate_seed,'sampled',n)
                    bank.save(folder/f'test_neg{n}_rep{policy_rep}.npz')
                    for model in MAIN_MODELS:
                        record_evaluation(fitted[model],b,bank,cfg,rec,seed,'candidate_sensitivity',dict(candidate_rep=policy_rep))
        if seed==cfg.seeds[-1]: anchor_state=(backbone,full_spec,full_search,sp_exp,banks)
        rec.save()
    if cfg.run_sensitivities:
        permutation_sensitivity(b,cfg,rec,*anchor_state)
        weighting_sensitivity(b,cfg,rec,*anchor_state)
    return selected


def quantile_order_ci(edges,q=.95,alpha=.05):
    from scipy.stats import binom
    x=np.sort(edges); n=len(x)
    lower=int(binom.ppf(alpha/2,n,q)); upper=int(binom.ppf(1-alpha/2,n,q))+1
    lo=float(x[lower-1]) if lower>=1 else -np.inf
    hi=float(x[upper-1]) if upper<=n else np.inf
    return lo,hi


def permutation_sensitivity(b,cfg,rec,backbone,spec,search,p,banks):
    t=b['train']; u=t.user.to_numpy(); i=t.item.to_numpy()
    mu,bu,bi=fit_bias(t,b['U'],b['I'],p['bias_reg']); r=t.rating.to_numpy(float)-mu-bu[u]-bi[i]
    uw,iw=vp_factors(b,cfg.vp_power,cfg.vp_smooth,cfg.factor_clip)
    seed=cfg.seeds[-1]
    for rep in range(cfg.permutation_repeats):
        st=time.perf_counter(); null_seed=seed+200001+rep*100003
        edges,checks,_=null_edges(t,r,uw,iw,(b['U'],b['I']),max(cfg.permutation_grid),null_seed,cfg)
        edge_seconds=time.perf_counter()-st
        for n in cfg.permutation_grid:
            sub=edges[:n]; edge=float(np.quantile(sub,cfg.edge_quantile)); lo,hi=quantile_order_ci(sub,cfg.edge_quantile)
            sm=spectral(b,cfg,seed,p,edge_override=edge)
            ff=search_fusion(backbone,sm,b,banks['valid_fusion'],cfg)
            mm=deploy_fusion(ff,cfg.gate_delta,'PermutationSensitivity')
            # Two analyses: fixed fusion coefficients isolates edge propagation;
            # reselected coefficients measures the complete pipeline response.
            fixed=copy.copy(mm)
            fixed.values=backbone.values.copy() if search.pre_gate_spectral_weight==search.pre_gate_popularity_weight==0 else (
                search.bpr_z+search.pre_gate_spectral_weight*fixed_normalize(sm.values,b['normalization_catalog'])+
                search.pre_gate_popularity_weight*search.popularity_z)
            for behavior,m in (('reselected_fusion',mm),('fixed_pre_gate_fusion',fixed)):
                agg,_=evaluate(m,banks['test'],b,cfg)
                rec.add('permutation_sensitivity',dataset=b['name'],model_seed=seed,rep=rep,null_seed=null_seed,
                    permutations=n,edge=edge,null_edge_sd=float(sub.std(ddof=1)),
                    quantile_ci_low=lo,quantile_ci_high=hi,quantile_ci_upper_finite=np.isfinite(hi),
                    detectable_rank=sm.metadata['detectable_rank'],retained_energy=sm.metadata['retained_energy'],
                    downstream_mode=behavior,shared_max_pool_seconds=edge_seconds,
                    approximate_prefix_cost_seconds=edge_seconds*n/max(cfg.permutation_grid),
                    max_solver_residual=float(checks[:n].max()),**mm.metadata,**agg)
        for j,x in enumerate(edges):
            rec.add('sensitivity_null_draws',dataset=b['name'],rep=rep,null_seed=null_seed,index=j,value=x)


def weighting_sensitivity(b,cfg,rec,backbone,spec,search,p,banks):
    # One fixed model seed + several weighting strengths; main matched NoVP
    # contrast above is repeated over all evidence seeds.
    seed=cfg.seeds[-1]
    for power in cfg.weighting_grid:
        sm=spectral(b,cfg,seed,p,power=power)
        ff=search_fusion(backbone,sm,b,banks['valid_fusion'],cfg)
        mm=deploy_fusion(ff,cfg.gate_delta,'WeightingSensitivity')
        agg,_=evaluate(mm,banks['test'],b,cfg)
        rec.add('weighting_sensitivity',dataset=b['name'],seed=seed,power=power,
            empirical_q_edge=sm.metadata['empirical_q_edge'],detectable_rank=sm.metadata['detectable_rank'],
            **mm.metadata,**agg)


def holm_adjust(pvalues):
    p=np.asarray(pvalues,float); order=np.argsort(p); result=np.empty(len(p)); running=0.
    for j,idx in enumerate(order):
        running=max(running,(len(p)-j)*p[idx]); result[idx]=min(1.,running)
    return result


def bootstrap_crossed(delta,reps,seed):
    """Crossed users x model seeds; both dimensions paired across methods.

    This accounts for these two variation sources without treating S*N rows
    as independent. With five seeds, uncertainty estimates remain approximate.
    """
    d=np.asarray(delta,float); S,N=d.shape; rng=np.random.default_rng(seed)
    values=np.empty(reps); user_values=np.empty(reps); averaged=d.mean(0)
    for k in range(reps):
        si=rng.integers(S,size=S); ui=rng.integers(N,size=N)
        values[k]=d[np.ix_(si,ui)].mean(); user_values[k]=averaged[ui].mean()
    return (*np.quantile(values,[.025,.975]),*np.quantile(user_values,[.025,.975]))


def paired_randomization(delta,reps,seed):
    d=np.asarray(delta,float); observed=abs(float(d.mean()))
    if np.allclose(d,0.,atol=1e-15,rtol=0): return 1.
    rng=np.random.default_rng(seed); extreme=0
    for st in range(0,reps,128):
        count=min(128,reps-st); signs=rng.choice((-1.,1.),size=(count,len(d)))
        extreme+=int(np.sum(np.abs((signs*d).mean(1))>=observed-1e-15))
    return (extreme+1)/(reps+1)


def rank_biserial(d):
    d=np.asarray(d); d=d[np.abs(d)>1e-15]
    if not len(d): return 0.
    r=rankdata(np.abs(d)); return float(np.sum(np.sign(d)*r)/r.sum())


def build_statistics(cfg,rec):
    metrics=rec.table('metrics'); main=metrics[metrics.stage=='main'].copy()
    metric_cols=[c for c in main if '@' in c]+['MRR','AUC','RMSE','MAE','MeanRank']
    for (dataset,model),g in main.groupby(['dataset','model']):
        for metric in metric_cols:
            x=pd.to_numeric(g[metric],errors='coerce').dropna().to_numpy()
            if not len(x): continue
            sd=float(np.std(x,ddof=1)) if len(x)>1 else np.nan
            half=float(student_t.ppf(.975,len(x)-1)*sd/np.sqrt(len(x))) if len(x)>1 else np.nan
            rec.add('main_summary',dataset=dataset,model=model,metric=metric,n_seeds=len(x),
                mean=x.mean(),sd=sd,seed_mean_ci_low=x.mean()-half,seed_mean_ci_high=x.mean()+half,
                ci_scope='t interval across model seeds on one fixed split/candidate bank')
    index=rec.table('detail_index'); paired=[]
    tested_metrics=[f'NDCG@{cfg.primary_k}',f'HR@{cfg.primary_k}',f'MRR@{cfg.primary_k}']
    for dataset in sorted(index.dataset.unique()):
        cache={}
        def load(model,seed):
            key=(model,seed)
            if key not in cache:
                rows=index[(index.dataset==dataset)&(index.model==model)&(index.seed==seed)]
                if len(rows)!=1: raise ValueError(f'Missing/duplicate detail for {key}')
                saved_path=Path(rows.iloc[0].path)
                path=saved_path if saved_path.is_absolute() else rec.root/saved_path
                if not path.exists():
                    filename=str(rows.iloc[0].path).replace('\\','/').split('/')[-1]
                    path=rec.root/dataset/'details'/filename
                cache[key]=pd.read_csv(path).sort_values(['user','target'])
            return cache[key]
        for a,b,contrast in CONTRASTS:
            for metric in tested_metrics:
                matrices=[]
                for seed in cfg.seeds:
                    da,db=load(a,seed),load(b,seed)
                    assert np.array_equal(da[['user','target']].to_numpy(),db[['user','target']].to_numpy())
                    assert da.candidate_hash.iloc[0]==db.candidate_hash.iloc[0]
                    matrices.append(da[metric].to_numpy()-db[metric].to_numpy())
                d=np.stack(matrices); avg=d.mean(0)
                lo,hi,ulo,uhi=bootstrap_crossed(d,cfg.bootstrap_reps,cfg.split_seed)
                sd=avg.std(ddof=1) if len(avg)>1 else 0.
                paired.append(dict(dataset=dataset,model_a=a,model_b=b,contrast=contrast,metric=metric,
                    n_seeds=len(d),n_users=d.shape[1],mean_delta=d.mean(),seed_delta_sd=d.mean(1).std(ddof=1),
                    crossed_ci_low=lo,crossed_ci_high=hi,conditional_user_ci_low=ulo,conditional_user_ci_high=uhi,
                    p_raw=paired_randomization(avg,cfg.randomization_reps,cfg.split_seed+117),
                    rank_biserial=rank_biserial(avg),paired_standardized_effect=avg.mean()/sd if sd>0 else 0.,
                    seed_wins=int(np.sum(d.mean(1)>1e-15)),seed_losses=int(np.sum(d.mean(1)<-1e-15)),
                    inference_scope='paired sign-flip over seed-averaged users; approximate shared-item dependence',
                    information_comparison='mixed information; not strict implicit parity' if contrast=='explicit_vs_implicit_reference' else contrast))
    df=pd.DataFrame(paired)
    for metric,idx in df.groupby('metric').groups.items():
        df.loc[idx,'p_holm']=holm_adjust(df.loc[idx,'p_raw'].to_numpy())
    df['significant_holm']=df.p_holm<.05
    df['evidence_mode']=cfg.mode
    df.to_csv(rec.root/'paired_tests.csv',index=False)
    rec.rows['paired_tests']=df.to_dict('records')
    state=rec.table('model_state')
    for (dataset,model),g in state.groupby(['dataset','model']):
        rec.add('activation_frequency',dataset=dataset,model=model,seeds=len(g),
            spectral_active=int((g.spectral_weight.abs()>1e-15).sum()),
            popularity_active=int((g.popularity_weight.abs()>1e-15).sum()),
            spectral_negative=int((g.spectral_weight<0).sum()),popularity_negative=int((g.popularity_weight<0).sum()),
            gate_rejections=int(g.gate_rejected.sum()),exact_backbone_fallbacks=int(g.exact_backbone_fallback.sum()))
    # Export a compact matrix of every metric and every model, not only the
    # primary cutoff. Missing rating metrics remain explicitly not applicable.
    summary=rec.table('main_summary')
    summary.assign(display=summary['mean'].map(lambda x:f'{x:.6f}')+' ± '+summary.sd.map(lambda x:f'{x:.6f}')).pivot_table(
        index=['dataset','model'],columns='metric',values='display',aggfunc='first').to_csv(rec.root/'evaluation_metric_matrix.csv')
    return main,summary,df


def gd_white_shrink(s,shape,sigma):
    """Gavish-Donoho Frobenius shrinker, ONLY complete i.i.d. white noise.

    Scale is sigma*sqrt(larger dimension), beta=smaller/larger dimension.
    It is not applied to sparse Amazon matrices or dependent-mask noise.
    """
    beta=min(shape)/max(shape); scale=sigma*math.sqrt(max(shape)); y=s/scale
    val=np.zeros_like(y); mask=y>1+math.sqrt(beta)
    val[mask]=np.sqrt(np.maximum((y[mask]**2-beta-1)**2-4*beta,0.))/y[mask]
    return val*scale


def synthetic_experiments(cfg,rec):
    shape=cfg.synthetic_shape; rank=min(5,min(shape)//4)
    densities=(1.,.1) if cfg.mode=='smoke' else (1.,.05,.1,.25)
    sigmas=(.3,) if cfg.mode=='smoke' else (.1,.3,.6)
    for rep in range(cfg.synthetic_repeats):
        for hetero,density,sigma in itertools.product(('uniform','user','item','both'),densities,sigmas):
            rng=np.random.default_rng(61001+rep)
            U,I=shape; P=rng.normal(size=(U,rank))/math.sqrt(rank); Q=rng.normal(size=(I,rank))/math.sqrt(rank)
            truth=P@Q.T
            pu=np.exp(rng.normal(0,.7,U)) if hetero in ('user','both') else np.ones(U)
            pi=np.exp(rng.normal(0,.7,I)) if hetero in ('item','both') else np.ones(I)
            pu/=pu.mean(); pi/=pi.mean()
            prob=np.ones(shape) if density==1 else np.clip(density*np.outer(pu,pi),.001,.8)
            mask=rng.random(shape)<prob; noise=rng.normal(0,sigma,shape)
            obs=mask*(truth+noise); u,i=np.where(mask)
            train=pd.DataFrame(dict(user=u,item=i,rating=obs[u,i]))
            b=dict(train=train,U=U,I=I,user_degree=np.bincount(u,minlength=U).astype(float),
                   item_degree=np.bincount(i,minlength=I).astype(float))
            target_observation=prob*truth
            rows=[]
            for power in (0.,cfg.vp_power):
                uw,iw=vp_factors(b,power,cfg.vp_smooth,cfg.factor_clip)
                X=sp.csr_matrix((obs[u,i]*uw[u]*iw[i],(u,i)),shape=shape)
                Uf,s,Vt,check=observed_svd(X,min(cfg.max_rank,min(shape)-1),61001+rep,cfg)
                edges,_,_=null_edges(train,obs[u,i],uw,iw,shape,cfg.permutations,71001+rep,cfg)
                edge=float(np.quantile(edges,cfg.edge_quantile))
                modes=('none','hard','soft','excess','reliability')
                for mode in modes:
                    sh=shrink(s,edge,mode); estimate=((Uf/uw[:,None])*sh)@(Vt/iw[None,:])
                    rows.append((f'{"NoVP" if power==0 else "VP"}_{mode}',estimate,edge,int(np.sum(s>edge)),check))
                if density==1 and power==0:
                    # Reference white-noise asymptotic model matches this case.
                    sh=gd_white_shrink(s,shape,sigma)
                    rows.append(('GD_white_reference', (Uf*sh)@Vt, np.nan,int(np.sum(sh>0)),check))
            rows.append(('Zero',np.zeros(shape),np.nan,0,{}))
            tu,_,_=np.linalg.svd(truth,full_matrices=False); tv,_,_=np.linalg.svd(truth.T,full_matrices=False)
            for name,est,edge,detected,check in rows:
                if np.linalg.norm(est)>1e-12:
                    eu,_,evt=np.linalg.svd(est,full_matrices=False)
                    subspace=(np.linalg.norm(tu[:,:rank].T@eu[:,:rank],'fro')**2+
                              np.linalg.norm(tv[:,:rank].T@evt.T[:,:rank],'fro')**2)/(2*rank)
                else: subspace=0.
                rec.add('synthetic_recovery',rep=rep,heterogeneity=hetero,observation_rate=density,
                    realized_density=float(mask.mean()),sigma=sigma,model=name,true_rank=rank,
                    detected_rank=detected,rank_absolute_error=abs(detected-rank),edge=edge,
                    relative_frobenius_to_signal=np.linalg.norm(est-truth)/np.linalg.norm(truth),
                    relative_frobenius_to_observation_expectation=np.linalg.norm(est-target_observation)/np.linalg.norm(target_observation),
                    unobserved_rmse=float(np.sqrt(np.mean((est[~mask]-truth[~mask])**2))) if (~mask).any() else np.nan,
                    subspace_overlap=subspace,
                    missingness_note='No oracle debiasing; raw full-signal recovery contains zero-filling bias.')
            # Noise-only conditional null: test finite-sample false detection.
            for power in (0.,cfg.vp_power):
                uw,iw=vp_factors(b,power,cfg.vp_smooth,cfg.factor_clip)
                x=noise[u,i]; H=sp.csr_matrix((x*uw[u]*iw[i],(u,i)),shape=shape)
                observed,check=top_one_verified(H,rep+81001,cfg.svd_tol)
                edges,_,_=null_edges(train,x,uw,iw,shape,cfg.permutations,rep+91001,cfg)
                edge=float(np.quantile(edges,cfg.edge_quantile))
                rec.add('synthetic_noise_calibration',rep=rep,heterogeneity=hetero,observation_rate=density,
                    sigma=sigma,power=power,observed_norm=observed,edge=edge,
                    false_detection=int(observed>edge),target_tail_probability=1-cfg.edge_quantile)
    recovery=rec.table('synthetic_recovery')
    recovery.groupby(['heterogeneity','observation_rate','sigma','model']).agg(
        repeats=('rep','nunique'),signal_error_mean=('relative_frobenius_to_signal','mean'),
        signal_error_sd=('relative_frobenius_to_signal','std'),
        observation_target_error_mean=('relative_frobenius_to_observation_expectation','mean'),
        subspace_overlap_mean=('subspace_overlap','mean'),rank_error_mean=('rank_absolute_error','mean')).to_csv(
            rec.root/'synthetic_recovery_summary.csv')
    cal=rec.table('synthetic_noise_calibration').groupby(['heterogeneity','observation_rate','sigma','power']).agg(
        repetitions=('rep','nunique'),false_detection_rate=('false_detection','mean'))
    n=cal.repetitions; p=cal.false_detection_rate; z=1.959963984540054
    center=(p+z*z/(2*n))/(1+z*z/n)
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)
    cal['wilson_ci_low']=center-half; cal['wilson_ci_high']=center+half
    cal['note']='Few replicates: broad CI; do not claim precise 5% false-detection calibration.'
    cal.to_csv(rec.root/'synthetic_noise_calibration_summary.csv')


REVIEWER_ROWS=[
 ('R1.1 / R2.1','Novelty','Integration/adaptation; no new RMT theorem or optimality claim','MANUSCRIPT_REVISION_AND_RESPONSES.md; references.json','Author literature comparison required'),
 ('R1.2','Recent-method rationale','LightGCN/LightGCL; distinguish observation calibration from graph representation','references.json; matched implicit/explicit comparisons','Manuscript rationale required'),
 ('R1.3 / R2.2','Degree justification','Conditional residual variance versus sparse-matrix energy; matched no/user/item/shuffled weighting','degree_diagnostics.csv; paired_tests.csv; synthetic_recovery_summary.csv','Implemented; empirical results pending paper run'),
 ('R1.4 / R2.3','Permutation stability','Nested 20/50/100/200/500 draws, independent null seeds, quantile CI, downstream fixed/reselected fusion','permutation_sensitivity.csv; sensitivity_null_draws.csv','Implemented; empirical results pending paper run'),
 ('R1.5 / R2.4','Shrinkage','Matched none/hard/soft/excess/reliability; white-noise GD reference on suitable synthetic cases','paired_tests.csv; singular_values.csv; synthetic_recovery_summary.csv','Implemented; empirical results pending paper run'),
 ('R1.6 / R2.8','NoGate and activation','Same pre-gate argmax; remove only margin. Joint gate; activation by signed nonzero coefficient','gate_sensitivity.csv; activation_frequency.csv; model_state.csv','Implemented; do not assume gate improves every domain'),
 ('R1.7 / R2.5','Modern baselines','Faithful LightGCN and LightGCL core; strong EASE/iALS controls; information parity and matched modern fusion','tuning_trials.csv; training_log.csv; paired_tests.csv','Implemented; compare learning curves and epoch caps'),
 ('R1.8 / R2.5','Budget and cost','12 base trials; same validation objective; frozen dataset configs; disclose extra fusion search and cold costs','tuning_budget.csv; runtime.csv; budget_summary.csv','Implemented; total capacities are disclosed, not claimed identical'),
 ('R1.9 / R2.10','Dataset origin','2014 source hints plus real SHA256, raw counts, author confirmation, complete preprocessing','data/manifest.json; provenance_checklist.csv','Author must confirm source identity/access record'),
 ('R1.10','Identical data/protocol','Single deduplicated graph; common chronological splits, exact candidate banks and hashes','data/*.csv; candidate_manifest.csv; integrity.csv','Implemented; new results replace old tables'),
 ('R1.11 / R2.7','Candidate sensitivity','Full catalog primary; nested sampled counts, capped counts explicit; fixed scoring','candidate_sensitivity_summary.csv; evaluation_metric_matrix.csv','Implemented; original 1000-negative claim corrected'),
 ('R1.12 / R2.9','Video Games/generalization','Signed weights, fallback, degree regimes, explicit/implicit null-family caveat','degree_regime_metrics.csv; activation_frequency.csv; evidence_claims.csv','Implemented evidence; interpretation depends on outcomes'),
 ('R1.13','Reduce repetition','Results=numbers; Discussion=mechanism/boundaries; Conclusion=supported contributions','MANUSCRIPT_REVISION_AND_RESPONSES.md','Original manuscript required for literal rewrite'),
 ('R1.14','Relevant JCTA literature','Only scientifically relevant references after full journal identity is confirmed','references.json; reviewer_response_matrix.csv','Author must provide full journal identity and select references'),
 ('R2.6','Statistical evidence','Every metric mean/SD/seed CI; paired ranking tests, crossed CI, effect size, Holm correction','main_summary.csv; paired_tests.csv; evaluation_metric_matrix.csv','Implemented; five-seed inference limits stated'),
 ('R2.10','Public reproducibility','Runnable notebook/module/CLI/tests/configs; executed dependency lock; public URL verification field','README.md; requirements-executed.txt; reproducibility_status.json','Publish repository and fill actual public URL before submission'),
]


REFERENCES=[
 dict(topic='LightGCN',url='https://arxiv.org/abs/2002.02126',year=2020),
 dict(topic='LightGCL; SVD collaborative contrastive learning',url='https://arxiv.org/abs/2302.08191',year=2023,
      code='https://github.com/HKUDS/LightGCL'),
 dict(topic='XSimGCL; additional recent related implicit method',url='https://arxiv.org/abs/2209.02544',journal_year=2024),
 dict(topic='OptShrink',url='https://arxiv.org/abs/1306.6042',year=2014),
 dict(topic='Whitening and heteroscedastic spectral shrinkage',url='https://arxiv.org/abs/1811.02201',journal_year=2021),
 dict(topic='eOptShrink; separable colored noise',url='https://doi.org/10.1016/j.acha.2024.101698',year=2025),
 dict(topic='Optimal shrinkage of singular values; white Gaussian reference',url='https://arxiv.org/abs/1405.7511',journal_year=2017),
 dict(topic='Sampled-metric inconsistency',url='https://www.ijcai.org/proceedings/2021/651',year=2021),
 dict(topic='Evaluation/tuning/reproducibility standards',url='https://doi.org/10.1145/3800587',year=2026),
 dict(topic='Amazon 2014 source',url='https://cseweb.ucsd.edu/~jmcauley/datasets/amazon/links.html',year=2014),
 dict(topic='EASE; strong linear implicit control, not a contemporary novelty claim',url='https://arxiv.org/abs/1905.03375',year=2019),
 dict(topic='Implicit ALS',url='https://doi.org/10.1109/ICDM.2008.22',year=2008),
]


def export_reviewer_matrix(rec):
    matrix=pd.DataFrame(REVIEWER_ROWS,columns=['reviewer_comments','issue','revision_design','evidence_files','status'])
    matrix.to_csv(rec.root/'reviewer_response_matrix.csv',index=False)
    experiment=[]
    for a,b,contrast in CONTRASTS:
        experiment.append(dict(experiment=contrast,model_a=a,model_b=b,
            datasets='health;toys;videogames',seeds='all evidence seeds',candidates='full catalog',
            primary='NDCG@10',secondary='HR/Recall/MRR/Precision@5,10,20; AUC; novelty; coverage; exposure',
            inference='paired users; crossed users x seeds CI; Holm correction',
            controlled='same data/split/candidates; mechanistic controls share fitted BPR and spectral configuration'))
    pd.DataFrame(experiment).to_csv(rec.root/'experiment_design_matrix.csv',index=False)
    rec.dump_json('references.json',REFERENCES)
    rec.dump_json('reproducibility_status.json',dict(
        benchmark_status='SMOKE_RUN_PENDING' if rec.cfg.mode=='smoke' else 'PAPER_RUN_PENDING',
        public_repository_url=rec.cfg.public_repository_url or None,
        public_repository_access_verified=False,
        jcta_full_name=rec.cfg.jcta_full_name or None,
        scientific_novelty_status='Integration/adaptation claim; no proof of entirely new component',
        outstanding=['Confirm dataset provenance','Publish and verify public repository',
                     'Provide full JCTA name and select relevant journal references',
                     'Apply numeric findings to actual manuscript; no original manuscript supplied']))


def export_findings(cfg,rec,summary,paired):
    primary=summary[summary.metric==f'NDCG@{cfg.primary_k}']
    claims=[]
    for dataset,g in primary[primary.model.isin(MAIN_MODELS)].groupby('dataset'):
        best=g.sort_values('mean',ascending=False).iloc[0]
        full=g[g.model=='VP_RMT_SVD'].iloc[0]
        implicit=g[g.model=='VP_RMT_Implicit'].iloc[0]
        claims.append(dict(dataset=dataset,best_model_by_mean=best.model,best_ndcg=best['mean'],
            proposed_full_mean=full['mean'],proposed_implicit_mean=implicit['mean'],
            evidence_level='illustrative_smoke' if cfg.mode=='smoke' else 'five-seed_fixed-split',
            permissible_claim=f"On {dataset}, {best.model} has the highest mean full-catalog NDCG@{cfg.primary_k} among the declared main models; pairwise significance must be read separately.",
            prohibited_claim='Universal superiority, calibrated probability from rho, or a new optimal RMT shrinker'))
    pd.DataFrame(claims).to_csv(rec.root/'evidence_claims.csv',index=False)
    candidate=rec.table('metrics'); candidate=candidate[candidate.stage=='candidate_sensitivity']
    if len(candidate):
        metric_cols=[c for c in candidate if '@' in c]+['AUC','CandidateMin','CandidateMedian','CandidateMax','CappedFraction']
        candidate.groupby(['dataset','model','requested_negatives'])[metric_cols].agg(['mean','std']).to_csv(
            rec.root/'candidate_sensitivity_summary.csv')
    perm=rec.table('permutation_sensitivity')
    if len(perm):
        perm.groupby(['dataset','permutations','downstream_mode']).agg(
            repeats=('rep','nunique'),edge_mean=('edge','mean'),edge_sd=('edge','std'),
            ndcg_mean=(f'NDCG@{cfg.primary_k}','mean'),ndcg_sd=(f'NDCG@{cfg.primary_k}','std')).to_csv(
                rec.root/'permutation_sensitivity_summary.csv')
    budgets=rec.table('tuning_budget'); runtime=rec.table('runtime')
    for (dataset,model),g in runtime[runtime.kind.str.startswith('standalone_cold')].groupby(['dataset','model']):
        matching=budgets[(budgets.dataset==dataset)&(budgets.model==model)]
        if model in ('VP_RMT_SVD','VP_RMT_Implicit','LightGCN_MatchedFusion'):
            sn='Spectral_Implicit' if model=='VP_RMT_Implicit' else 'Spectral_Explicit'
            bn='LightGCN' if model=='LightGCN_MatchedFusion' else 'BPRMF'
            matching=budgets[(budgets.dataset==dataset)&budgets.model.isin([bn,sn])]
        rec.add('budget_summary',dataset=dataset,model=model,
            dataset_tuning_seconds=float(matching.tuning_seconds.sum()),
            dataset_tuning_trials=int(matching.trials.sum()),
            cold_fit_mean_seconds=g.seconds.mean(),cold_fit_sd_seconds=g.seconds.std(),
            test_ranking_mean_seconds=rec.table('metrics').query('dataset == @dataset and model == @model and stage == "main"').ranking_evaluation_seconds.mean(),
            additional_fusion_grid=len(cfg.fusion_grid)**2 if model.startswith('VP_RMT') or model=='LightGCN_MatchedFusion' else 0,
            cost_scope='Tuning once per dataset; cold evidence fit independently; fusion selection is part of cold fit.')
    # Figures are generated from current run tables, never copied from old outputs.
    plt.figure(figsize=(10,4))
    for name,g in primary[primary.model.isin(('BPRMF','LightGCN','LightGCL','VP_RMT_Implicit','VP_RMT_SVD'))].groupby('model'):
        plt.errorbar(g.dataset,g['mean'],yerr=g.sd,marker='o',capsize=3,label=name)
    plt.ylabel(f'Full-catalog NDCG@{cfg.primary_k}'); plt.legend(fontsize=8); plt.tight_layout()
    if cfg.mode=='smoke': plt.title('SMOKE ONLY — GENERATED DATA; NOT PAPER RESULTS')
    plt.savefig(rec.root/'full_catalog_ndcg.png',dpi=180); plt.close()
    if len(perm):
        plt.figure(figsize=(7,4))
        for name,g in perm[perm.downstream_mode=='reselected_fusion'].groupby('dataset'):
            s=g.groupby('permutations').edge.agg(['mean','std'])
            plt.errorbar(s.index,s['mean'],yerr=s['std'],marker='o',capsize=3,label=name)
        plt.xlabel('Number of permutations'); plt.ylabel('Empirical 95% null edge'); plt.legend(); plt.tight_layout()
        if cfg.mode=='smoke': plt.title('SMOKE ONLY — GENERATED DATA')
        plt.savefig(rec.root/'permutation_stability.png',dpi=180); plt.close()
    make_html_report(cfg,rec,claims,paired)


def make_html_report(cfg,rec,claims,paired):
    import html
    marker='SMOKE TEST: ANGKA INI BUKAN HASIL PAPER' if cfg.mode=='smoke' else 'HASIL RUN PAPER; PERIKSA SEMUA CHECKLIST SEBELUM SUBMISSION'
    matrix=pd.DataFrame(REVIEWER_ROWS,columns=['Komentar','Isu','Desain perbaikan','Bukti','Status'])
    body=f'''<!doctype html><meta charset="utf-8"><title>VP-RMT Revision Evidence</title>
    <style>body{{font:15px system-ui;max-width:1200px;margin:36px auto;line-height:1.6;padding:20px;color:#172336}}table{{border-collapse:collapse;width:100%;font-size:13px}}td,th{{padding:9px;border:1px solid #d5dce4;text-align:left}}th{{background:#edf2f7}}.flag{{background:#fff1ce;padding:16px}}</style>
    <h1>VP-RMT-SVD: evidence untuk revisi reviewer</h1><p class="flag">{marker}</p>
    <p>Full-item evaluation, normalisasi fusion tetap dari katalog TRAIN, dua partisi validation pengguna yang disjoint,
    konfigurasi baseline dipilih satu kali per dataset lalu dibekukan, serta gate margin dipisahkan dari pemilihan bobot fusion.</p>
    <h2>Klaim numerik yang dihasilkan</h2>{pd.DataFrame(claims).to_html(index=False,escape=True)}
    <h2>Matriks reviewer</h2>{matrix.to_html(index=False,escape=True)}
    <h2>Pasangan utama dan ablation</h2>{paired.to_html(index=False,escape=True)}
    <h2>Landasan metodologis dan batas klaim</h2>
    <p>Komponen SVD, degree normalization, shrinkage, BPR, dan validation selection berasal dari teknik mapan.
    Kontribusi yang dinilai adalah adaptasi finite-sample observation-aware calibration dan integrasi yang terkontrol.
    Rho=s²/(s²+edge²) adalah bobot shrinkage heuristik, bukan probabilitas reliability terkalibrasi atau estimator optimal yang telah dibuktikan.</p>
    <p>Untuk X=Oε dengan noise mean-zero dan mask independen, Var(X_ui)=p_ui Var(ε_ui).
    Dengan demikian heterogenitas matriks sparse berbeda dari heteroskedastisitas residual bersyarat pada observasi.
    Model p_ui sebanding degree pengguna × degree item adalah asumsi exposure separabel, bukan hukum untuk data MNAR.
    Power 0.5 merupakan partial normalization; sensitivitas dan kontrol power=0 menguji manfaatnya.</p>
    <p>Clipping dilakukan pada faktor baris/kolom agar inverse diagonal konsisten. Empirical q95 edge dan shrinkage scale dipisahkan;
    detectable rank selalu memakai edge yang tidak diskalakan. Continuous shrinkage mempertahankan komponen di bawah edge.</p>
    <p>Null explicit mempertahankan mask dan mengacak nilai residual; struktur mask tetap ada. Null implicit mengacak item stubs,
    mempertahankan degree stub dan memungkinkan multiedges yang dicatat. Karena null family berbeda, full-vs-implicit bukan ablation murni sumber informasi.</p>
    <p>NoGate menghilangkan margin saja dari argmax yang sama; grid tetap memuat backbone. Hasil identik ketika gate tidak menolak adalah hasil yang sah.
    Delta=0.0005 diperlakukan sebagai margin praktis yang ditetapkan sebelum test, bukan signifikansi statistik.
    Nilai threshold/sensitivity tidak boleh dipilih kembali berdasarkan hasil test.</p>
    <p>Recall@K=HR@K dan Precision@K=HR@K/K untuk satu positif LOO. AUC terhadap unobserved candidates bukan bukti bahwa seluruh kandidat adalah negatif benar.
    Novelty, tail share, coverage, dan exposure Gini mengukur karakteristik rekomendasi; bukan fairness demografis.</p>
    <p>CI crossed meresampling seed dan pengguna secara berpasangan. Uji sign-flip memakai perbedaan pengguna setelah averaging seed,
    dengan Holm per metrik atas seluruh pasangan/dataset. Dependensi shared item dan hanya lima seed membatasi inferensi.</p>
    <p>Filtrasi 11-core menghasilkan subset pengguna aktif; chronological LOO per pengguna bukan global temporal split.
    Tiga kategori Amazon tidak membuktikan generalisasi antar-platform, cold-start, maupun online performance.</p>
    <h2>Petunjuk revisi naskah</h2><p>Results: tampilkan tabel mean±SD, CI, efek, biaya, sensitivitas, dan activation.
    Discussion: jelaskan mekanisme yang benar-benar didukung, pengaruh popularity, perbedaan domain, dan batas generalisasi.
    Conclusion: nyatakan kontribusi integrasi serta domain keberhasilan/kegagalan; hilangkan klaim superioritas universal.
    Naskah asli belum disediakan sehingga dokumen ini adalah panduan evidence, bukan pengganti edit literal manuskrip.</p>
    <h2>Referensi primer</h2><ul>'''
    for r in REFERENCES:
        body+=f'<li>{html.escape(r["topic"])}: <a href="{html.escape(r["url"])}">{html.escape(r["url"])}</a></li>'
    body+='</ul><p>Identitas JCTA, referensi JCTA yang relevan, konfirmasi sumber CSV, dan URL repository publik harus diisi oleh penulis sebelum submission.</p>'
    (rec.root/'reviewer_evidence_report.html').write_text(body,encoding='utf-8')


def final_integrity(cfg,rec):
    metrics=rec.table('metrics'); main=metrics[metrics.stage=='main']
    rows=[]
    for dataset,g in main.groupby('dataset'):
        for model in MAIN_MODELS:
            ss=set(g[g.model==model].seed.astype(int))
            rows.append(dict(check=f'{dataset}/{model}/all_seeds',passed=ss==set(cfg.seeds)))
        rows.append(dict(check=f'{dataset}/identical_full_candidate_hash',passed=g.candidate_hash.nunique()==1))
    rows.append(dict(check='full_primary_protocol',passed=bool((main.policy=='full').all())))
    rows.append(dict(check='finite_primary_metrics',passed=bool(np.isfinite(main[f'NDCG@{cfg.primary_k}']).all())))
    rows.append(dict(check='no_failed_tuning',passed=bool((rec.table('tuning_trials').status=='ok').all())))
    frame=pd.DataFrame(rows); frame.to_csv(rec.root/'integrity.csv',index=False)
    if not frame.passed.all(): raise AssertionError('Experiment integrity checks failed')
    return frame


def run_all(config,bundles=None):
    start=time.perf_counter(); cfg=config.resolved()
    inputs=None
    if bundles is None:
        inputs=check_data_files(cfg)
        # Freeze the exact files so every method uses the same resolved inputs.
        cfg.data_files.update(dict(zip(inputs.dataset,inputs.resolved_path)))
    rec=Recorder(cfg)
    if inputs is not None: inputs.to_csv(rec.root/'input_file_check.csv',index=False)
    export_reviewer_matrix(rec)
    data_start=time.perf_counter()
    if bundles is None:
        bundles={name:load_bundle(name,cfg,rec) for name in cfg.datasets}
    rec.add('pipeline_cost',stage='load_preprocess_manifest_candidates_excluded',seconds=time.perf_counter()-data_start)
    for b in bundles.values(): run_dataset(b,cfg,rec)
    synthetic_start=time.perf_counter()
    if cfg.run_synthetic: synthetic_experiments(cfg,rec)
    rec.add('pipeline_cost',stage='synthetic_experiments',seconds=time.perf_counter()-synthetic_start)
    statistics_start=time.perf_counter()
    _,summary,paired=build_statistics(cfg,rec)
    export_findings(cfg,rec,summary,paired); integrity=final_integrity(cfg,rec); rec.save()
    rec.add('pipeline_cost',stage='statistics_reports_integrity',seconds=time.perf_counter()-statistics_start)
    try:
        import resource
        peak_rss=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    except ImportError:
        peak_rss=None
    rec.add('pipeline_cost',stage='total_including_all_tuning_controls_sensitivities_and_reports',
        seconds=time.perf_counter()-start,
        process_lifetime_peak_rss_native_units=peak_rss,
        peak_rss_units='bytes on macOS; KiB on Linux; unavailable on Windows; lifetime, not per-model allocation',
        cuda_lifetime_peak_allocated_bytes=torch.cuda.max_memory_allocated() if torch is not None and torch.cuda.is_available() else None)
    status=json.loads((rec.root/'reproducibility_status.json').read_text())
    status['benchmark_status']='SMOKE_ONLY_NOT_PAPER_EVIDENCE' if cfg.mode=='smoke' else 'PAPER_RUN_COMPLETED'
    status['integrity_checks_passed']=True
    rec.dump_json('reproducibility_status.json',status); rec.save()
    print('Completed:',rec.root.resolve(),flush=True)
    print('Evidence status:', 'SMOKE ONLY' if cfg.mode=='smoke' else 'PAPER RUN',flush=True)
    return rec,summary,paired,integrity


## Cek teknis dan dataset

Objek `CFG` dibuat di sini, setelah semua fungsi dan kelas didefinisikan.
Tiga file yang sudah terdeteksi pada notebook sebelumnya dipilih secara eksplisit
agar folder tidak berubah dan tidak tertukar dengan file lain. Pemeriksaan
berhenti sebelum training jika salah satu file tidak dapat dibaca dari kernel ini.


In [3]:
def quick_checks():
    # Equal scores must follow target-independent item tie priorities.
    t=pd.DataFrame({'user':[0,1], 'item':[4,3]})
    h=pd.DataFrame({'user':[0,1], 'item':[0,0]})
    bank=CandidateBank(t,h,5,91,'full')
    b={'U':2,'I':5,'user_degree':np.ones(2),'item_degree':np.ones(5)}
    cfg=Config(ks=(1,3,5),primary_k=3)
    _,d=evaluate(Scores(np.zeros((2,5)),'ties','implicit_edges'),bank,b,cfg)
    for row in d.itertuples():
        eligible=np.arange(1,5)
        expected=1+np.sum(bank.tie_priority[eligible]<bank.tie_priority[row.target])
        assert row.rank==expected
    A=np.random.default_rng(8).normal(size=(9,7))
    norm,res=top_one_verified(sp.csr_matrix(A),123)
    assert np.isclose(norm,np.linalg.svd(A,compute_uv=False)[0],rtol=1e-7)
    assert res<1e-6
    print('Quick numerical/evaluator checks: PASS. Not a dataset benchmark.')
    return {'tie_policy':'PASS','spectral_norm':'PASS','evidence':'TECHNICAL_CHECK_ONLY'}

quick_checks()
require_torch()

CFG = Config()
CFG.data_files = {name: DATASET_SPECS[name]['filename'] for name in CFG.datasets}

DATA_CHECK = check_data_files(CFG)
display(DATA_CHECK[['dataset', 'status', 'resolved_path', 'bytes']])
display(pd.DataFrame([{
    'Mode': CFG.mode,
    'Dataset': ', '.join(CFG.datasets),
    'Seed': ', '.join(map(str, CFG.seeds)),
    'Trials per tunable component': CFG.n_trials,
    'Permutations utama': CFG.permutations,
    'Primary metric': f'NDCG@{CFG.primary_k} (full-item)',
    'Device': torch_device(CFG),
    'Output': CFG.output_root,
}]))


Quick numerical/evaluator checks: PASS. Not a dataset benchmark.
Kernel: Linux | Working directory: /mnt/g/SVD
CFG.data_dir: /mnt/g/SVD/dataset
   dataset status                     resolved_path
    health  FOUND     /mnt/g/SVD/dataset/health.csv
      toys  FOUND       /mnt/g/SVD/dataset/toys.csv
videogames  FOUND /mnt/g/SVD/dataset/VideoGames.csv


,dataset,status,resolved_path,bytes
0,health,FOUND,/mnt/g/SVD/dataset/health.csv,121507623
1,toys,FOUND,/mnt/g/SVD/dataset/toys.csv,91781923
2,videogames,FOUND,/mnt/g/SVD/dataset/VideoGames.csv,53949503


,Mode,Dataset,Seed,Trials per tunable component,Permutations utama,Primary metric,Device,Output
0,paper,"health, toys, videogames","2021, 2022, 2023, 2024, 2025",12,200,NDCG@10 (full-item),cuda,/mnt/g/SVD/vp_rmt_reviewer_revision_runs


## Jalankan semua eksperimen paper

Sel berikut memakai `CFG` yang sudah diperiksa. Urutannya otomatis: preprocessing
bersama → tuning → lima seed/model dan ablation → sensitivitas → synthetic
recovery → statistik → tabel, figure, laporan HTML dan integrity checks.
Setiap run membuat folder hasil baru. Progres dicetak per dataset dan seed.


In [4]:
REC, SUMMARY, PAIRED, INTEGRITY = run_all(CFG)

print("SELESAI. Hasil disimpan di:", REC.root.resolve())
print("Laporan reviewer:", (REC.root / 'reviewer_evidence_report.html').resolve())


Kernel: Linux | Working directory: /mnt/g/SVD
CFG.data_dir: /mnt/g/SVD/dataset
   dataset status                     resolved_path
    health  FOUND     /mnt/g/SVD/dataset/health.csv
      toys  FOUND       /mnt/g/SVD/dataset/toys.csv
videogames  FOUND /mnt/g/SVD/dataset/VideoGames.csv
[health] tuning on 1176 users; fusion on 785 other users
[health] evidence seed 2021
[health] evidence seed 2022
[health] evidence seed 2023
[health] evidence seed 2024
[health] evidence seed 2025
[toys] tuning on 472 users; fusion on 316 other users
[toys] evidence seed 2021
[toys] evidence seed 2022
[toys] evidence seed 2023
[toys] evidence seed 2024
[toys] evidence seed 2025
[videogames] tuning on 1086 users; fusion on 724 other users
[videogames] evidence seed 2021
[videogames] evidence seed 2022
[videogames] evidence seed 2023
[videogames] evidence seed 2024
[videogames] evidence seed 2025
Completed: /mnt/g/SVD/vp_rmt_reviewer_revision_runs/20261006T045433Z_73c3766a
Evidence status: PAPER RUN
SELESA

## Matriks hasil, statistik dan evidence reviewer

Tabel lengkap dan figure juga tersimpan otomatis di folder run. Mean dan SD
tersedia untuk semua metrik; paired CI/effect/tests diberikan untuk 25 contrasts
terencana. Recall=HR pada evaluasi satu-positive LOO. RMSE/MAE model fusion
merupakan metrik secondary spectral rating head.


In [5]:
display(INTEGRITY)
display(pd.read_csv(REC.root / 'evaluation_metric_matrix.csv'))
display(PAIRED)
display(REC.table('activation_frequency'))
display(REC.table('budget_summary'))
display(REC.table('pipeline_cost'))

for filename in (
    'reviewer_response_matrix.csv',
    'permutation_sensitivity_summary.csv',
    'candidate_sensitivity_summary.csv',
    'provenance_checklist.csv',
    'evidence_claims.csv',
):
    path = REC.root / filename
    if path.exists():
        print("\n" + filename)
        display(pd.read_csv(path))


,check,passed
0,health/Popularity/all_seeds,True
1,health/PureSVD_Implicit/all_seeds,True
2,health/PureSVD_Ratings/all_seeds,True
3,health/BPRMF/all_seeds,True
4,health/LightGCN/all_seeds,True
5,health/LightGCL/all_seeds,True
6,health/iALS/all_seeds,True
7,health/EASE/all_seeds,True
8,health/VP_RMT_SVD/all_seeds,True
9,health/VP_RMT_Implicit/all_seeds,True


,dataset,model,AUC,CatalogCoverage@10,CatalogCoverage@20,CatalogCoverage@5,ExposureGini@10,ExposureGini@20,ExposureGini@5,HR@10,...,Precision@10,Precision@20,Precision@5,RMSE,Recall@10,Recall@20,Recall@5,TailShare@10,TailShare@20,TailShare@5
0,health,BPRMF,0.471473 ± 0.012843,0.373232 ± 0.106598,0.516667 ± 0.116646,0.263973 ± 0.087365,0.931870 ± 0.025653,0.894002 ± 0.036582,0.954445 ± 0.018709,0.064661 ± 0.003293,...,0.006466 ± 0.000329,0.004064 ± 0.000098,0.009689 ± 0.000563,NaN,0.064661 ± 0.003293,0.081285 ± 0.001962,0.048445 ± 0.002816,0.017807 ± 0.000946,0.023590 ± 0.001215,0.012545 ± 0.002028
1,health,BPR_Pop,0.392807 ± 0.005965,0.152862 ± 0.020837,0.237710 ± 0.034622,0.097138 ± 0.014550,0.976045 ± 0.002696,0.958418 ± 0.004147,0.986004 ± 0.001597,0.078633 ± 0.000838,...,0.007863 ± 0.000084,0.004595 ± 0.000033,0.013687 ± 0.000221,NaN,0.078633 ± 0.000838,0.091892 ± 0.000665,0.068434 ± 0.001106,0.015502 ± 0.000036,0.020617 ± 0.000264,0.008445 ± 0.000845
2,health,BPR_Spectral,0.511888 ± 0.011450,0.230135 ± 0.036081,0.340572 ± 0.049440,0.150842 ± 0.024616,0.962695 ± 0.006795,0.936273 ± 0.010646,0.977170 ± 0.004386,0.079245 ± 0.001469,...,0.007925 ± 0.000147,0.005023 ± 0.000107,0.012585 ± 0.000246,0.856755 ± 0.000494,0.079245 ± 0.001469,0.100459 ± 0.002133,0.062927 ± 0.001228,0.016084 ± 0.000175,0.021968 ± 0.000220,0.009771 ± 0.000839
3,health,EASE,0.467562 ± 0.000000,0.163300 ± 0.000000,0.243266 ± 0.000000,0.106902 ± 0.000000,0.974923 ± 0.000000,0.956878 ± 0.000000,0.984668 ± 0.000000,0.075472 ± 0.000000,...,0.007547 ± 0.000000,0.004564 ± 0.000000,0.013157 ± 0.000000,NaN,0.075472 ± 0.000000,0.091280 ± 0.000000,0.065783 ± 0.000000,0.016318 ± 0.000000,0.020755 ± 0.000000,0.010607 ± 0.000000
4,health,LightGCL,0.442987 ± 0.002902,0.192929 ± 0.009190,0.309428 ± 0.011703,0.128788 ± 0.005615,0.965850 ± 0.001566,0.944174 ± 0.000788,0.976709 ± 0.001263,0.060173 ± 0.002816,...,0.006017 ± 0.000282,0.003794 ± 0.000177,0.007506 ± 0.000602,NaN,0.060173 ± 0.002816,0.075880 ± 0.003540,0.037532 ± 0.003008,0.017124 ± 0.000441,0.023085 ± 0.000496,0.010260 ± 0.000811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,videogames,VP_Soft,0.744041 ± 0.014680,0.371623 ± 0.083870,0.533697 ± 0.114660,0.254434 ± 0.059617,0.899570 ± 0.028048,0.850571 ± 0.038464,0.930571 ± 0.020030,0.064751 ± 0.004251,...,0.006475 ± 0.000425,0.005127 ± 0.000172,0.008000 ± 0.000168,1.064963 ± 0.000036,0.064751 ± 0.004251,0.102541 ± 0.003441,0.040000 ± 0.000838,0.006497 ± 0.004104,0.012133 ± 0.006726,0.003006 ± 0.002296
92,videogames,VP_Soft_Spectral,0.506240 ± 0.000025,0.013097 ± 0.000305,0.028922 ± 0.000374,0.008186 ± 0.000000,0.992868 ± 0.000001,0.985827 ± 0.000001,0.996353 ± 0.000003,0.022099 ± 0.000000,...,0.002210 ± 0.000000,0.001657 ± 0.000000,0.003315 ± 0.000000,1.064963 ± 0.000036,0.022099 ± 0.000000,0.033149 ± 0.000000,0.016575 ± 0.000000,0.000000 ± 0.000000,0.000000 ± 0.000000,0.000000 ± 0.000000
93,videogames,VP_UserOnly,0.752522 ± 0.012171,0.443656 ± 0.093661,0.624011 ± 0.115783,0.304229 ± 0.064956,0.876008 ± 0.028667,0.818955 ± 0.040347,0.914125 ± 0.019166,0.065746 ± 0.003965,...,0.006575 ± 0.000396,0.005171 ± 0.000177,0.008243 ± 0.000266,1.058099 ± 0.000367,0.065746 ± 0.003965,0.103425 ± 0.003546,0.041215 ± 0.001331,0.010309 ± 0.006584,0.018580 ± 0.010392,0.004994 ± 0.003774
94,videogames,VP_UserOnly_Spectral,0.510160 ± 0.000303,0.043656 ± 0.000835,0.068895 ± 0.002686,0.030014 ± 0.001276,0.992323 ± 0.000012,0.984729 ± 0.000027,0.995612 ± 0.000018,0.022652 ± 0.000391,...,0.002265 ± 0.000039,0.001641 ± 0.000031,0.002453 ± 0.000121,1.058099 ± 0.000367,0.022652 ± 0.000391,0.032818 ± 0.000630,0.012265 ± 0.000605,0.000000 ± 0.000000,0.000055 ± 0.000000,0.000000 ± 0.000000


,dataset,model_a,model_b,contrast,metric,n_seeds,n_users,mean_delta,seed_delta_sd,crossed_ci_low,...,p_raw,rank_biserial,paired_standardized_effect,seed_wins,seed_losses,inference_scope,information_comparison,p_holm,significant_holm,evidence_mode
0,health,VP_RMT_SVD,BPRMF,overall_vs_backbone,NDCG@10,5,1961,0.027971,0.000912,0.022017,...,0.000200,0.776178,0.211745,5,0,paired sign-flip over seed-averaged users; app...,overall_vs_backbone,0.014997,True,paper
1,health,VP_RMT_SVD,BPRMF,overall_vs_backbone,HR@10,5,1961,0.017338,0.003515,0.010913,...,0.000200,0.645389,0.134913,5,0,paired sign-flip over seed-averaged users; app...,overall_vs_backbone,0.014997,True,paper
2,health,VP_RMT_SVD,BPRMF,overall_vs_backbone,MRR@10,5,1961,0.031257,0.001021,0.024662,...,0.000200,0.783432,0.214378,5,0,paired sign-flip over seed-averaged users; app...,overall_vs_backbone,0.014997,True,paper
3,health,VP_RMT_SVD,LightGCN,explicit_vs_implicit_reference,NDCG@10,5,1961,0.002084,0.004764,-0.002442,...,0.125575,0.187667,0.034335,3,2,paired sign-flip over seed-averaged users; app...,mixed information; not strict implicit parity,1.000000,False,paper
4,health,VP_RMT_SVD,LightGCN,explicit_vs_implicit_reference,HR@10,5,1961,0.006527,0.000756,0.002040,...,0.005599,0.512195,0.064494,5,0,paired sign-flip over seed-averaged users; app...,mixed information; not strict implicit parity,0.313537,False,paper
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,videogames,VP_RMT_SVD_Spectral,VP_Soft_Spectral,spectral_only_shrinkage,HR@10,5,1810,-0.000221,0.000303,-0.001105,...,1.000000,-0.333333,-0.014863,0,2,paired sign-flip over seed-averaged users; app...,spectral_only_shrinkage,1.000000,False,paper
221,videogames,VP_RMT_SVD_Spectral,VP_Soft_Spectral,spectral_only_shrinkage,MRR@10,5,1810,0.001173,0.000344,-0.000040,...,0.063987,0.247863,0.042781,5,0,paired sign-flip over seed-averaged users; app...,spectral_only_shrinkage,1.000000,False,paper
222,videogames,VP_RMT_SVD_Spectral,VP_Excess_Spectral,spectral_only_shrinkage,NDCG@10,5,1810,0.000730,0.000251,-0.000242,...,0.132973,0.225071,0.034916,5,0,paired sign-flip over seed-averaged users; app...,spectral_only_shrinkage,1.000000,False,paper
223,videogames,VP_RMT_SVD_Spectral,VP_Excess_Spectral,spectral_only_shrinkage,HR@10,5,1810,-0.000221,0.000303,-0.001105,...,1.000000,-0.333333,-0.014863,0,2,paired sign-flip over seed-averaged users; app...,spectral_only_shrinkage,1.000000,False,paper


,dataset,model,seeds,spectral_active,popularity_active,spectral_negative,popularity_negative,gate_rejections,exact_backbone_fallbacks,evidence_mode
0,health,BPR_Pop,5,0,5,0,0,0,0,paper
1,health,BPR_Spectral,5,5,0,0,0,0,0,paper
2,health,LightGCN_MatchedFusion,5,5,5,0,5,0,0,paper
3,health,VP_Excess,5,5,5,0,0,0,0,paper
4,health,VP_Hard,5,5,5,0,0,0,0,paper
5,health,VP_ItemOnly,5,5,5,0,0,0,0,paper
6,health,VP_NoGate,5,5,5,0,0,0,0,paper
7,health,VP_NoShrink,5,5,5,0,0,0,0,paper
8,health,VP_NoVP,5,5,5,0,0,0,0,paper
9,health,VP_RMT_Implicit,5,5,5,0,0,0,0,paper


,dataset,model,dataset_tuning_seconds,dataset_tuning_trials,cold_fit_mean_seconds,cold_fit_sd_seconds,test_ranking_mean_seconds,additional_fusion_grid,cost_scope,evidence_mode
0,health,BPRMF,158.264099,12,16.698627,4.487347,0.394915,0,Tuning once per dataset; cold evidence fit ind...,paper
1,health,EASE,2.040241,12,0.099855,0.028574,0.390805,0,Tuning once per dataset; cold evidence fit ind...,paper
2,health,LightGCL,639.383571,12,26.873032,0.502548,0.388095,0,Tuning once per dataset; cold evidence fit ind...,paper
3,health,LightGCN,171.980791,12,11.215535,1.462320,0.378218,0,Tuning once per dataset; cold evidence fit ind...,paper
4,health,LightGCN_MatchedFusion,195.660960,24,18.450860,1.778445,0.395700,81,Tuning once per dataset; cold evidence fit ind...,paper
5,health,Popularity,0.021737,1,0.002344,0.000532,0.372444,0,Tuning once per dataset; cold evidence fit ind...,paper
6,health,PureSVD_Implicit,2.559348,12,0.190553,0.063029,0.439283,0,Tuning once per dataset; cold evidence fit ind...,paper
7,health,PureSVD_Ratings,2.618847,12,0.170245,0.146700,0.431565,0,Tuning once per dataset; cold evidence fit ind...,paper
8,health,VP_RMT_Implicit,170.831020,24,20.199371,2.965757,0.378513,81,Tuning once per dataset; cold evidence fit ind...,paper
9,health,VP_RMT_SVD,181.944268,24,19.827821,2.643091,0.379734,81,Tuning once per dataset; cold evidence fit ind...,paper


,stage,seconds,evidence_mode,process_lifetime_peak_rss_native_units,peak_rss_units,cuda_lifetime_peak_allocated_bytes
0,load_preprocess_manifest_candidates_excluded,29.275200,paper,NaN,NaN,NaN
1,synthetic_experiments,1003.205608,paper,NaN,NaN,NaN
2,statistics_reports_integrity,39.236663,paper,NaN,NaN,NaN
3,total_including_all_tuning_controls_sensitivit...,7213.912789,paper,3162244.0,bytes on macOS; KiB on Linux; unavailable on W...,81112576.0


... (6 outputs left out)


provenance_checklist.csv


,dataset,category,local_path,raw_sha256,raw_rows,version_hint,hint_source_url,hint_download_url,raw_count_matches_hint,author_confirmed,author_record,note,evidence_mode
0,health,Health and Personal Care,/mnt/g/SVD/dataset/health.csv,387374ea5c9641a6e9dc09b93423414da4f8153b297559...,2982326,Amazon 2014,https://cseweb.ucsd.edu/~jmcauley/datasets/ama...,https://snap.stanford.edu/data/amazon/productG...,True,False,{},"Counts and filenames are hints, not proof of s...",paper
1,toys,Toys and Games,/mnt/g/SVD/dataset/toys.csv,e28a342ca89a79036a9a840c0385a3a3342a08d3d3450e...,2252771,Amazon 2014,https://cseweb.ucsd.edu/~jmcauley/datasets/ama...,https://snap.stanford.edu/data/amazon/productG...,True,False,{},"Counts and filenames are hints, not proof of s...",paper
2,videogames,Video Games,/mnt/g/SVD/dataset/VideoGames.csv,f0ca171050c2e0f208d47f5267992476b21ab68a01439c...,1324753,Amazon 2014,https://cseweb.ucsd.edu/~jmcauley/datasets/ama...,https://snap.stanford.edu/data/amazon/productG...,True,False,{},"Counts and filenames are hints, not proof of s...",paper



evidence_claims.csv


,dataset,best_model_by_mean,best_ndcg,proposed_full_mean,proposed_implicit_mean,evidence_level,permissible_claim,prohibited_claim
0,health,Popularity,0.066123,0.065113,0.052155,five-seed_fixed-split,"On health, Popularity has the highest mean ful...","Universal superiority, calibrated probability ..."
1,toys,VP_RMT_Implicit,0.047697,0.021935,0.047697,five-seed_fixed-split,"On toys, VP_RMT_Implicit has the highest mean ...","Universal superiority, calibrated probability ..."
2,videogames,iALS,0.038697,0.033649,0.036846,five-seed_fixed-split,"On videogames, iALS has the highest mean full-...","Universal superiority, calibrated probability ..."


## Menggunakan hasil untuk revisi paper

Gunakan hasil CSV asli dari run ini untuk mengganti tabel/figure lama. Highest
mean tidak otomatis menunjukkan keunggulan signifikan: baca paired CI, effect
size dan Holm-corrected tests. NoGate dapat identik jika margin tidak menolak;
weighting/shrinkage dapat memberi hasil yang berbeda antar-dataset.

Novelty diposisikan sebagai integrasi/adaptasi. Konfirmasi sumber/version CSV,
URL GitHub publik, nama lengkap JCTA dan referensi jurnal yang relevan sebelum
submission. Identitas sumber tidak dibuktikan hanya oleh nama/lokasi file.
Panduan evidence dan seluruh pemetaan reviewer dihasilkan pada laporan HTML.
